# 01 · The cohort, the outcome ladder and the 21 predictors

This notebook turns the survey file into the study's first analysis frame: one row per woman
whose most recent birth has a recorded number of antenatal care contacts, the outcome
ladder, the survey design, and the 21 predictors the study built by hand from the survey's
variables. Each predictor has its own section: the raw variable is audited, and the decision
on how to code it (which categories to merge, what missing means, what "not applicable"
means) is written down before it is made.

The code below is the study's own, cell for cell; only the file paths are set once, here.
It writes `outputs/ANC8_analysis_frozen.pkl` (the analysis frame), the predictor and outcome
registers, and a descriptive Table 1. The last cell checks the frame against the study's.

**Reads:** `data/ETIR8AFL.DTA` · **Writes:** `outputs/`

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

from anc8 import cohorts
PROJECT = P.project
print("survey file:", cohorts.IR_FILE)

# 1. ANC8 outcome and predictor ascertainment

## 1.1 Analysis purpose

This notebook develops the analytic cohort, outcome definitions, predictor
constructs, and preprocessing rules for a national machine-learning analysis
of antenatal care contact attainment using the 2024–25 Ethiopia Demographic
and Health Survey Individual Recode dataset.

The primary outcome is attainment of eight or more antenatal care contacts
(ANC8+). Attainment of four or more contacts (ANC4+) is retained as a
secondary benchmark outcome.

The analysis is designed from a national representativeness perspective.
Survey weights, primary sampling units, and stratification variables are
therefore retained throughout data preparation for subsequent descriptive
and predictive analyses.

Predictor definitions previously validated against the EDHS documentation
will be reused where scientifically appropriate. However, all predictors will
be reconstructed and re-audited within the ANC analytic population because
category distributions, structural non-applicability, missingness, and
modelling suitability may differ from those observed in earlier analytic
cohorts.

This notebook performs no model fitting. Its purpose is to produce a
documented, reproducible, and fixed analytic dataset and accompanying
registers for subsequent machine-learning readiness and prediction notebooks.

In [ ]:
# ======================================================================
# LOAD THE ORIGINAL 2024–25 EDHS INDIVIDUAL RECODE DATA
#
# Purpose:
# Load the original source dataset for the ANC8 analysis and verify its
# basic structure before any eligibility restriction or variable
# construction is performed.
#
# No cohort definition, outcome construction, or preprocessing occurs
# in this cell.
# ======================================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ----------------------------------------------------------------------
# Project directory
# ----------------------------------------------------------------------

PROJECT_ROOT = PROJECT


# ----------------------------------------------------------------------
# Original EDHS Individual Recode file
# ----------------------------------------------------------------------

IR_FILE = cohorts.IR_FILE


# ----------------------------------------------------------------------
# Load source data
# ----------------------------------------------------------------------

ir_df = pd.read_stata(
    IR_FILE,
    convert_categoricals=False
)


# ----------------------------------------------------------------------
# Basic source-data audit
# ----------------------------------------------------------------------

print("Project root:", PROJECT_ROOT)
print("IR file:", IR_FILE)

print("\nDataset shape:", ir_df.shape)
print("Number of rows:", len(ir_df))
print("Number of columns:", ir_df.shape[1])

print("\ncaseid present:", "caseid" in ir_df.columns)

if "caseid" in ir_df.columns:
    print("caseid unique:", ir_df["caseid"].is_unique)

## 1.2 Maternal-health roster and ANC contact variable

The ANC contact outcomes are based on `m14_1`, the number of antenatal care
visits reported for the most recent eligible pregnancy covered by the
maternal-health roster.

Before constructing ANC4+ or ANC8+, the variable is audited in the original
IR dataset to establish:

- the number of women covered by the maternal-health roster;
- the number reporting no ANC;
- the distribution of reported ANC contacts; and
- the number with unknown contact counts.

Women outside the maternal-health roster are structurally not eligible for
these outcome variables and are not treated as ordinary missing observations.

In [ ]:
# ======================================================================
# AUDIT NUMBER OF ANC CONTACTS
#
# Purpose:
# Examine m14_1 directly in the original EDHS IR dataset before defining
# the ANC analytic cohort or constructing ANC4+ and ANC8+ outcomes.
#
# Coding expected from the validated EDHS mapping:
#   0  = no ANC contacts
#   1+ = reported number of ANC contacts
#   98 = don't know
#   NaN = outside the relevant maternal-health roster / not applicable
# ======================================================================

print("m14_1 distribution:")
print(
    ir_df["m14_1"]
    .value_counts(dropna=False)
    .sort_index()
)

# ----------------------------------------------------------------------
# Basic roster accounting
# ----------------------------------------------------------------------

n_total = len(ir_df)

n_mh_roster = ir_df["m14_1"].notna().sum()
n_no_anc = ir_df["m14_1"].eq(0).sum()
n_known_positive_contacts = ir_df["m14_1"].between(1, 97).sum()
n_unknown_contacts = ir_df["m14_1"].eq(98).sum()
n_outside_roster = ir_df["m14_1"].isna().sum()

print("\nRoster accounting")
print("-----------------")
print("Total IR respondents:", n_total)
print("Maternal-health roster records:", n_mh_roster)
print("No ANC contacts:", n_no_anc)
print("Known ANC contact count >=1:", n_known_positive_contacts)
print("Unknown ANC contact count (98):", n_unknown_contacts)
print("Outside maternal-health roster:", n_outside_roster)

# ----------------------------------------------------------------------
# Internal accounting check
# ----------------------------------------------------------------------

print(
    "\nAccounting check:",
    n_mh_roster
    == n_no_anc + n_known_positive_contacts + n_unknown_contacts
)

## 1.3 ANC contact attainment outcomes

For ANC4+ and ANC8+, women reporting zero ANC contacts are retained in the
analytic denominator and classified as not attaining either threshold.

Women coded `98 = don't know` for the number of ANC contacts are treated as
unresolved because their attainment status cannot be determined.

The primary outcome denominator therefore includes all maternal-health roster
records with a known ANC contact count from 0 to 97.

In [ ]:
# ======================================================================
# DEFINE ANC4+ AND ANC8+ OUTCOMES
#
# Purpose:
# Construct the two ANC contact-attainment outcomes using women with a
# known number of ANC contacts.
#
# Definitions:
#   ANC4+ = 1 if m14_1 >= 4, otherwise 0
#   ANC8+ = 1 if m14_1 >= 8, otherwise 0
#
# m14_1 = 98 is unresolved and excluded from the resolved denominator.
# ======================================================================

anc_outcome_df = ir_df.loc[
    ir_df["m14_1"].notna(),
    [
        "caseid",
        "m14_1",
        "v005",
        "v021",
        "v022"
    ]
].copy()

# ----------------------------------------------------------------------
# Resolve known contact counts.
# ----------------------------------------------------------------------

known_contacts = anc_outcome_df["m14_1"].between(0, 97)

anc_outcome_df["anc4plus"] = np.where(
    known_contacts,
    (anc_outcome_df["m14_1"] >= 4).astype(int),
    np.nan
)

anc_outcome_df["anc8plus"] = np.where(
    known_contacts,
    (anc_outcome_df["m14_1"] >= 8).astype(int),
    np.nan
)

# ----------------------------------------------------------------------
# Audit outcome distributions.
# ----------------------------------------------------------------------

for outcome in ["anc4plus", "anc8plus"]:

    print("\n" + "=" * 60)
    print(outcome)
    print("=" * 60)

    print(
        anc_outcome_df[outcome]
        .value_counts(dropna=False)
    )

    resolved = anc_outcome_df[outcome].notna()

    weighted_prev = np.average(
        anc_outcome_df.loc[resolved, outcome],
        weights=anc_outcome_df.loc[resolved, "v005"] / 1_000_000
    )

    print(
        "Resolved denominator:",
        resolved.sum()
    )

    print(
        "Unweighted prevalence:",
        round(
            anc_outcome_df.loc[resolved, outcome].mean(),
            4
        )
    )

    print(
        "Survey-weighted prevalence:",
        round(
            weighted_prev,
            4
        )
    )

For both ANC4+ and ANC8+, women with `m14_1 = 98` were classified as
unresolved because the reported number of ANC contacts was unknown.
These records were retained in the outcome dataset but coded as missing
(`NaN`) for the derived binary outcomes and excluded from resolved-outcome
denominators.

## 1.4 Predictor ascertainment framework

Candidate predictors are identified from the previously validated EDHS
predictor mapping register and reconstructed within the ANC analytic cohort.

For the primary ANC8+ prediction model, predictor eligibility is guided by
temporality and intended use. Variables should represent characteristics that
could reasonably be known before or early in pregnancy and should not directly
encode subsequent ANC utilization or care received.

Predictors are therefore re-evaluated in the ANC cohort with attention to:

- source-variable validity;
- applicability to the ANC analytic population;
- structural non-applicability;
- special response codes;
- missingness;
- category frequencies;
- potential redundancy; and
- risk of temporal or conceptual leakage.

Previously validated mappings are reused where appropriate, but category
collapsing and final modelling representations are determined from the ANC
cohort rather than copied mechanically from earlier analyses.

In [ ]:
# ======================================================================
# DEFINE CANDIDATE BACKGROUND PREDICTORS FOR ANC8+ ASCERTAINMENT
#
# Purpose:
# Establish the initial predictor set from previously validated EDHS
# mappings before any recoding or modelling decisions are made.
#
# No category collapsing or missing-value treatment occurs in this cell.
# ======================================================================

candidate_predictor_vars = [
    # Maternal and sociodemographic
    "v106",      # maternal education
    "v501",      # marital status
    "v717",      # maternal occupation
    "v701",      # partner education
    "v705",      # partner occupation

    # Household and socioeconomic
    "v190",      # wealth quintile
    "v025",      # residence
    "v024",      # region
    "v136",      # household size
    "v130",      # religion

    # Media / connectivity / protection
    "v157",      # newspaper exposure
    "v158",      # radio exposure
    "v159",      # television exposure
    "v171a",     # internet exposure
    "v481",      # health insurance

    # Pregnancy and reproductive history
    "m10_1",     # pregnancy intention
    "bord_01",   # birth order
    "b11_01",    # preceding birth interval

    # Access
    "s1101"      # travel time to nearest hospital / health center
]

predictor_audit_df = anc_outcome_df[
    ["caseid", "anc8plus", "anc4plus"]
].merge(
    ir_df[
        ["caseid"] + candidate_predictor_vars
    ],
    on="caseid",
    how="left",
    validate="one_to_one"
)

print("Predictor audit dataset shape:", predictor_audit_df.shape)

print("\nMissingness:")
print(
    predictor_audit_df[candidate_predictor_vars]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

## 1.5 ANC8-specific predictor eligibility register

The previously fixed EDHS predictor mapping register is retained as the
authoritative source for variable identification, coding structure, roster
applicability, special-response handling, and previously validated derived
constructs.

Because the prediction target has changed from post-delivery transition
protection to ANC contact attainment, each previously mapped construct is
reassessed for the ANC8 study according to its temporal relationship with the
outcome.

Each construct is assigned one of the following ANC8-specific roles:

1. **Primary candidate**  
   Plausibly available before or early in pregnancy and eligible for the
   primary ANC8 prediction model.

2. **Secondary/process predictor**  
   Occurs during ANC and may be scientifically useful for a secondary
   prediction model, but is excluded from the primary background-risk model.

3. **Excluded: downstream**  
   Occurs at delivery, after delivery, or otherwise after ANC contact
   accumulation and therefore cannot predict ANC8 attainment prospectively.

4. **Outcome source**  
   Directly defines ANC4+ or ANC8+ and cannot be used as a predictor.

5. **Survey-design/support variable**  
   Retained for weighting, clustering, stratification, splitting, or
   evaluation but never used as a predictor.

Final inclusion and coding remain subject to empirical audit within the ANC
analytic cohort.

In [ ]:
# ======================================================================
# CREATE ANC8-SPECIFIC PREDICTOR ELIGIBILITY REGISTER
#
# Purpose:
# Translate previously validated EDHS mappings into roles appropriate
# for the new ANC8 prediction question.
#
# IMPORTANT:
# This register does NOT replace the fixed EDHS mapping register.
# It adds an ANC8-specific temporality / eligibility decision layer.
#
# No variables are constructed in this cell.
# ======================================================================

anc8_predictor_register = pd.DataFrame(
    [
        # --------------------------------------------------------------
        # Maternal / sociodemographic
        # --------------------------------------------------------------
        {
            "construct": "Maternal age at index birth",
            "source": "v011 + b3_01",
            "anc8_role": "Primary candidate",
            "reason": "Maternal characteristic established before ANC contact accumulation."
        },
        {
            "construct": "Maternal education",
            "source": "v106",
            "anc8_role": "Primary candidate",
            "reason": "Pre-existing maternal socioeconomic characteristic."
        },
        {
            "construct": "Marital / union status",
            "source": "v501",
            "anc8_role": "Primary candidate",
            "reason": "Background maternal characteristic."
        },
        {
            "construct": "Maternal occupation",
            "source": "v717",
            "anc8_role": "Primary candidate",
            "reason": "Background socioeconomic characteristic."
        },
        {
            "construct": "Partner education",
            "source": "v701",
            "anc8_role": "Primary candidate",
            "reason": "Background household characteristic; structural non-applicability must be preserved."
        },
        {
            "construct": "Partner occupation",
            "source": "v705",
            "anc8_role": "Primary candidate",
            "reason": "Background household characteristic; structural non-applicability must be preserved."
        },
        {
            "construct": "Healthcare decision-making participation",
            "source": "v743a",
            "anc8_role": "Primary candidate",
            "reason": "Potential pre-existing autonomy/access characteristic; structural non-applicability must be preserved."
        },

        # --------------------------------------------------------------
        # Household / socioeconomic
        # --------------------------------------------------------------
        {
            "construct": "Household wealth quintile",
            "source": "v190",
            "anc8_role": "Primary candidate",
            "reason": "Household socioeconomic characteristic."
        },
        {
            "construct": "Place of residence",
            "source": "v025",
            "anc8_role": "Primary candidate",
            "reason": "Geographic/contextual characteristic."
        },
        {
            "construct": "Region",
            "source": "v024",
            "anc8_role": "Primary candidate",
            "reason": "National geographic/contextual characteristic."
        },
        {
            "construct": "Household size",
            "source": "v136",
            "anc8_role": "Primary candidate",
            "reason": "Household characteristic."
        },
        {
            "construct": "Religion",
            "source": "v130",
            "anc8_role": "Primary candidate",
            "reason": "Background sociodemographic characteristic."
        },
        {
            "construct": "Traditional-media exposure",
            "source": "v157 + v158 + v159",
            "anc8_role": "Primary candidate",
            "reason": "Background information-access characteristic."
        },
        {
            "construct": "Internet exposure",
            "source": "v171a",
            "anc8_role": "Primary candidate",
            "reason": "Background information-access characteristic."
        },
        {
            "construct": "Health insurance coverage",
            "source": "v481",
            "anc8_role": "Primary candidate",
            "reason": "Pre-existing financial-access characteristic."
        },

        # --------------------------------------------------------------
        # Reproductive / pregnancy history
        # --------------------------------------------------------------
        {
            "construct": "Birth order",
            "source": "bord_01",
            "anc8_role": "Primary candidate",
            "reason": "Reproductive history known before completion of ANC contacts."
        },
        {
            "construct": "Preceding birth interval",
            "source": "b11_01",
            "anc8_role": "Primary candidate",
            "reason": "Prior reproductive-history characteristic; structurally not applicable to first births."
        },
        {
            "construct": "Pregnancy intention",
            "source": "m10_1",
            "anc8_role": "Primary candidate",
            "reason": "Characteristic of the index pregnancy conceptually preceding ANC utilization."
        },
        {
            "construct": "Previous adverse pregnancy outcome",
            "source": "p32_* linked through pidxb_*",
            "anc8_role": "Primary candidate",
            "reason": "Prior reproductive history; requires validated respondent-specific linkage."
        },

        # --------------------------------------------------------------
        # Access
        # --------------------------------------------------------------
        {
            "construct": "Travel time to nearest hospital / health center",
            "source": "s1101",
            "anc8_role": "Primary candidate",
            "reason": "Pre-existing geographic access characteristic."
        },
        {
            "construct": "Transport mode to nearest hospital / health center",
            "source": "s1101a",
            "anc8_role": "Primary candidate",
            "reason": "Pre-existing geographic access characteristic."
        },

        # --------------------------------------------------------------
        # ANC-process variables
        # --------------------------------------------------------------
        {
            "construct": "Timing of first ANC",
            "source": "m13_1 / m13a_1",
            "anc8_role": "Secondary/process predictor",
            "reason": "Occurs during the ANC pathway; excluded from the primary background-risk model."
        },
        {
            "construct": "ANC provider",
            "source": "m2*_1",
            "anc8_role": "Secondary/process predictor",
            "reason": "Observed after ANC has begun."
        },
        {
            "construct": "ANC source / place",
            "source": "m57*_1",
            "anc8_role": "Secondary/process predictor",
            "reason": "Observed after ANC has begun."
        },
        {
            "construct": "ANC clinical content",
            "source": "m42c_1:m42f_1",
            "anc8_role": "Secondary/process predictor",
            "reason": "Care received during ANC."
        },
        {
            "construct": "ANC counselling content",
            "source": "m42g_1:m43_1",
            "anc8_role": "Secondary/process predictor",
            "reason": "Care received during ANC."
        },
        {
            "construct": "Iron supplementation",
            "source": "m45_1 / m46_1",
            "anc8_role": "Secondary/process predictor",
            "reason": "Pregnancy care/intervention occurring during the ANC period."
        },

        # --------------------------------------------------------------
        # Direct outcome
        # --------------------------------------------------------------
        {
            "construct": "Number of ANC contacts",
            "source": "m14_1",
            "anc8_role": "Outcome source",
            "reason": "Directly defines ANC4+ and ANC8+."
        },

        # --------------------------------------------------------------
        # Survey-design support
        # --------------------------------------------------------------
        {
            "construct": "Survey weight",
            "source": "v005",
            "anc8_role": "Survey-design/support variable",
            "reason": "Used for nationally representative estimation and weighted evaluation; never a predictor."
        },
        {
            "construct": "Primary sampling unit",
            "source": "v021",
            "anc8_role": "Survey-design/support variable",
            "reason": "Used for cluster-aware splitting and survey design; never a predictor."
        },
        {
            "construct": "Sampling stratum",
            "source": "v022",
            "anc8_role": "Survey-design/support variable",
            "reason": "Retained for survey-design analyses; never a predictor."
        }
    ]
)

print("ANC8-specific eligibility register:")
print(
    anc8_predictor_register[
        ["construct", "source", "anc8_role"]
    ].to_string(index=False)
)

print("\nRole counts:")
print(
    anc8_predictor_register["anc8_role"]
    .value_counts()
)

## 1.6 Raw audit of primary candidate predictors

Before constructing or collapsing predictors, the directly observed source
variables for primary ANC8 candidates are audited within the maternal-health
roster.

The audit is intended to identify:

- variable availability;
- ordinary missingness;
- structural non-applicability;
- special response codes;
- sparse categories; and
- potential inconsistencies requiring construct-specific review.

Derived constructs such as maternal age at index birth and previous adverse
pregnancy outcome are not created in this step. They will be handled
separately using their validated source-variable rules.

In [ ]:
# ======================================================================
# RAW AUDIT OF DIRECT PRIMARY-CANDIDATE SOURCE VARIABLES
#
# Purpose:
# Examine the raw coding and completeness of directly observed candidate
# predictors within the 6,916 maternal-health roster records.
#
# IMPORTANT:
# - No recoding occurs here.
# - No category collapsing occurs here.
# - Structural non-applicability is not automatically treated as ordinary
#   missingness.
# - Derived constructs are handled in later dedicated cells.
# ======================================================================

direct_primary_sources = [
    "v106",      # maternal education
    "v501",      # marital / union status
    "v717",      # maternal occupation
    "v701",      # partner education
    "v705",      # partner occupation
    "v743a",     # healthcare decision-making participation

    "v190",      # wealth quintile
    "v025",      # residence
    "v024",      # region
    "v136",      # household size
    "v130",      # religion

    "v157",      # newspaper exposure
    "v158",      # radio exposure
    "v159",      # television exposure
    "v171a",     # internet exposure
    "v481",      # health insurance

    "bord_01",   # birth order
    "b11_01",    # preceding birth interval
    "m10_1",     # pregnancy intention

    "s1101",     # travel time
    "s1101a"     # transport mode
]

# ----------------------------------------------------------------------
# Check that all listed source variables exist in the IR dataset.
# ----------------------------------------------------------------------

missing_from_ir = [
    var for var in direct_primary_sources
    if var not in ir_df.columns
]

print("Variables missing from IR dataset:", missing_from_ir)


# ----------------------------------------------------------------------
# Restrict raw audit to the maternal-health roster.
# ----------------------------------------------------------------------

anc_primary_raw = anc_outcome_df[
    ["caseid", "anc4plus", "anc8plus"]
].merge(
    ir_df[
        ["caseid"] + direct_primary_sources
    ],
    on="caseid",
    how="left",
    validate="one_to_one"
)

print("\nRaw primary-candidate audit shape:", anc_primary_raw.shape)


# ----------------------------------------------------------------------
# Produce a compact completeness summary.
# ----------------------------------------------------------------------

audit_summary = pd.DataFrame({
    "variable": direct_primary_sources,
    "non_missing_n": [
        anc_primary_raw[var].notna().sum()
        for var in direct_primary_sources
    ],
    "missing_n": [
        anc_primary_raw[var].isna().sum()
        for var in direct_primary_sources
    ],
    "unique_non_missing": [
        anc_primary_raw[var].nunique(dropna=True)
        for var in direct_primary_sources
    ]
})

audit_summary["missing_pct"] = (
    audit_summary["missing_n"] / len(anc_primary_raw) * 100
).round(2)

print("\nCompleteness summary:")
print(
    audit_summary
    .sort_values(["missing_n", "variable"], ascending=[False, True])
    .to_string(index=False)
)

### 1.6.1 Partner and healthcare decision-making variables

Partner education, partner occupation, and healthcare decision-making are
potentially useful predictors of ANC contact attainment, but their apparent
missingness may reflect structural non-applicability among women not currently
in union rather than ordinary missing data.

The raw coding of `v501`, `v701`, `v705`, and `v743a` is therefore examined
jointly before any recoding is performed.

In [ ]:
# ======================================================================
# AUDIT PARTNER AND HEALTHCARE DECISION-MAKING VARIABLES
#
# Purpose:
# Determine whether missing values in partner education, partner
# occupation, and healthcare decision-making are structurally related
# to marital / union status.
#
# No recoding is performed in this cell.
# ======================================================================

partner_vars = [
    "v501",   # current marital / union status
    "v701",   # partner education
    "v705",   # partner occupation
    "v743a"   # person who usually decides on respondent's healthcare
]

# ----------------------------------------------------------------------
# Display raw distributions.
# ----------------------------------------------------------------------

for var in partner_vars:
    print("\n" + "=" * 60)
    print(var)
    print("=" * 60)

    print(
        anc_primary_raw[var]
        .value_counts(dropna=False)
        .sort_index()
    )


# ----------------------------------------------------------------------
# Examine missingness by marital / union status.
# ----------------------------------------------------------------------

for var in ["v701", "v705", "v743a"]:

    print("\n" + "=" * 60)
    print(f"Missingness in {var} by v501")
    print("=" * 60)

    table = pd.crosstab(
        anc_primary_raw["v501"],
        anc_primary_raw[var].isna(),
        margins=True
    )

    table.columns = ["Observed", "Missing", "Total"]

    print(table)

### 1.6.1 Decision on partner and autonomy variables

Missingness in partner education (`v701`) and partner occupation (`v705`) is
entirely explained by marital/union status and represents structural
non-applicability rather than ordinary missing data.

Healthcare decision-making (`v743a`) shows the same structural pattern, with
one additional missing observation among women for whom the variable is
otherwise applicable.

Accordingly:

- partner education and partner occupation will retain an explicit
  `Not applicable / not currently in union` category;
- healthcare decision-making will retain the same structural category, while
  its single remaining missing value will remain distinct as `Unknown/missing`;
- no structural non-applicable value will be imputed as if it were ordinary
  missing data.

In [ ]:
# ======================================================================
# RECODE PARTNER AND HEALTHCARE DECISION-MAKING VARIABLES
#
# Purpose:
# Preserve structural non-applicability explicitly rather than treating
# these values as ordinary missing data.
# ======================================================================

partner_recode = anc_primary_raw.copy()

# ----------------------------------------------------------------------
# Identify women for whom partner/autonomy variables are structurally
# not applicable based on marital / union status.
# ----------------------------------------------------------------------

not_currently_partnered = partner_recode["v501"].isin([0, 3, 4, 5])

# ----------------------------------------------------------------------
# Partner education
# ----------------------------------------------------------------------

partner_recode["partner_education"] = (
    partner_recode["v701"]
    .astype("object")
)

partner_recode.loc[
    not_currently_partnered & partner_recode["v701"].isna(),
    "partner_education"
] = "Not applicable / not currently in union"


# ----------------------------------------------------------------------
# Partner occupation
# ----------------------------------------------------------------------

partner_recode["partner_occupation"] = (
    partner_recode["v705"]
    .astype("object")
)

partner_recode.loc[
    not_currently_partnered & partner_recode["v705"].isna(),
    "partner_occupation"
] = "Not applicable / not currently in union"


# ----------------------------------------------------------------------
# Healthcare decision-making
# ----------------------------------------------------------------------

partner_recode["healthcare_decision_making"] = (
    partner_recode["v743a"]
    .astype("object")
)

partner_recode.loc[
    not_currently_partnered & partner_recode["v743a"].isna(),
    "healthcare_decision_making"
] = "Not applicable / not currently in union"

# Preserve the one genuinely unresolved applicable value separately.
partner_recode.loc[
    (~not_currently_partnered) &
    partner_recode["v743a"].isna(),
    "healthcare_decision_making"
] = "Unknown / missing"


# ----------------------------------------------------------------------
# Readable audit output
# ----------------------------------------------------------------------

label_map = {
    "partner_education": "v701 = Partner education",
    "partner_occupation": "v705 = Partner occupation",
    "healthcare_decision_making": "v743a = Healthcare decision-making"
}

for var, label in label_map.items():

    print("\n" + "=" * 70)
    print(label)
    print("=" * 70)

    print(
        partner_recode[var]
        .value_counts(dropna=False)
    )

### 1.6.2 Preceding birth interval

`b11_01` represents the interval preceding the most recent live birth.

This construct is structurally not applicable to first births because there is
no preceding live birth from which an interval can be calculated.

The raw relationship between `b11_01` and `bord_01` is therefore examined
before constructing grouped birth-interval categories. Any missing values among
higher-order births will be reviewed separately rather than automatically
treated as structural non-applicability.

In [ ]:
# ======================================================================
# AUDIT PRECEDING BIRTH INTERVAL AGAINST BIRTH ORDER
#
# Purpose:
# Determine whether missingness in b11_01 is structurally explained by
# first birth status and identify any residual missingness among women
# with birth order >= 2.
#
# No grouping or recoding is performed in this cell.
# ======================================================================

print("\n" + "=" * 70)
print("bord_01 = Birth order of most recent live birth")
print("=" * 70)

print(
    anc_primary_raw["bord_01"]
    .value_counts(dropna=False)
    .sort_index()
)


print("\n" + "=" * 70)
print("b11_01 = Preceding birth interval")
print("=" * 70)

print(
    anc_primary_raw["b11_01"]
    .value_counts(dropna=False)
    .sort_index()
)


# ----------------------------------------------------------------------
# Examine missingness in preceding birth interval by birth order.
# ----------------------------------------------------------------------

birth_interval_missing_by_order = pd.crosstab(
    anc_primary_raw["bord_01"],
    anc_primary_raw["b11_01"].isna(),
    margins=True
)

birth_interval_missing_by_order.columns = [
    "b11_01 observed",
    "b11_01 missing",
    "Total"
]

print("\n" + "=" * 70)
print("b11_01 missingness by bord_01 = Birth order")
print("=" * 70)

print(birth_interval_missing_by_order)


# ----------------------------------------------------------------------
# Explicitly count structural and residual missingness.
# ----------------------------------------------------------------------

first_birth = anc_primary_raw["bord_01"].eq(1)

n_structural_first_birth_missing = (
    first_birth & anc_primary_raw["b11_01"].isna()
).sum()

n_missing_later_birth = (
    anc_primary_raw["bord_01"].ge(2) &
    anc_primary_raw["b11_01"].isna()
).sum()

n_birth_order_missing = anc_primary_raw["bord_01"].isna().sum()

print("\nMissingness interpretation")
print("--------------------------")
print(
    "Missing b11_01 among first births:",
    n_structural_first_birth_missing
)
print(
    "Missing b11_01 among birth order >= 2:",
    n_missing_later_birth
)
print(
    "Missing bord_01 itself:",
    n_birth_order_missing
)

### 1.6.2 Preceding birth interval: preprocessing decision

Missingness in `b11_01` is predominantly structural.

All 1,491 women with birth order 1 have no preceding birth interval, as expected.
These observations are therefore represented explicitly as `First birth` rather
than treated as missing.

Among women with birth order 2 or higher, 17 records have no reported preceding
birth interval and are retained separately as `Unknown / missing`.

Twelve records have missing birth order and cannot be reliably classified as
first birth or higher-order birth. These are also retained as `Unknown / missing`.

For modelling, preceding birth interval is grouped into clinically interpretable
categories while preserving structural non-applicability:

- First birth
- <24 months
- 24–35 months
- 36–59 months
- 60+ months
- Unknown / missing

In [ ]:
# ======================================================================
# CONSTRUCT GROUPED PRECEDING BIRTH INTERVAL
#
# Purpose:
# Convert b11_01 into an interpretable categorical predictor while
# explicitly preserving:
#
#   1. First births as structural non-applicability
#   2. Genuine/residual missingness as "Unknown / missing"
#
# Raw variable:
#   b11_01 = preceding birth interval in months
#
# Derived categories:
#   First birth
#   <24 months
#   24-35 months
#   36-59 months
#   60+ months
#   Unknown / missing
# ======================================================================

birth_interval_recode = anc_primary_raw.copy()

# ----------------------------------------------------------------------
# Start with all observations classified as unknown.
# This prevents unclassified records from being silently lost.
# ----------------------------------------------------------------------

birth_interval_recode["preceding_birth_interval_group"] = (
    "Unknown / missing"
)

# ----------------------------------------------------------------------
# First birth:
# No preceding live birth exists, so b11_01 is structurally not
# applicable rather than ordinarily missing.
# ----------------------------------------------------------------------

birth_interval_recode.loc[
    birth_interval_recode["bord_01"].eq(1),
    "preceding_birth_interval_group"
] = "First birth"

# ----------------------------------------------------------------------
# Higher-order births with an observed interval.
# b11_01 is measured in months.
# ----------------------------------------------------------------------

birth_interval_recode.loc[
    birth_interval_recode["b11_01"].between(0, 23),
    "preceding_birth_interval_group"
] = "<24 months"

birth_interval_recode.loc[
    birth_interval_recode["b11_01"].between(24, 35),
    "preceding_birth_interval_group"
] = "24-35 months"

birth_interval_recode.loc[
    birth_interval_recode["b11_01"].between(36, 59),
    "preceding_birth_interval_group"
] = "36-59 months"

birth_interval_recode.loc[
    birth_interval_recode["b11_01"].ge(60),
    "preceding_birth_interval_group"
] = "60+ months"


# ----------------------------------------------------------------------
# Make category order explicit for readable output and later modelling.
# ----------------------------------------------------------------------

interval_order = [
    "First birth",
    "<24 months",
    "24-35 months",
    "36-59 months",
    "60+ months",
    "Unknown / missing"
]

birth_interval_recode["preceding_birth_interval_group"] = pd.Categorical(
    birth_interval_recode["preceding_birth_interval_group"],
    categories=interval_order,
    ordered=False
)


# ----------------------------------------------------------------------
# Readable audit output
# ----------------------------------------------------------------------

print("\n" + "=" * 72)
print("b11_01 = Preceding birth interval in months")
print("Derived predictor = Preceding birth interval group")
print("=" * 72)

print("\nCategory definitions:")
print("  First birth       = no preceding live birth; structurally not applicable")
print("  <24 months        = preceding interval 0-23 months")
print("  24-35 months      = preceding interval 24-35 months")
print("  36-59 months      = preceding interval 36-59 months")
print("  60+ months        = preceding interval >=60 months")
print("  Unknown / missing = residual missing interval or missing birth order")

print("\nDerived distribution:")
print(
    birth_interval_recode["preceding_birth_interval_group"]
    .value_counts(dropna=False)
    .reindex(interval_order)
)

print(
    "\nTotal:",
    birth_interval_recode["preceding_birth_interval_group"].notna().sum()
)

### 1.6.3 Birth order

`bord_01` records the birth order of the most recent live birth. The fixed
predictor register previously validated this measure against total children ever
born (`v201`) and confirmed observed values ranging from 1 to 17.

The ANC analytic cohort is first examined using the original count variable.
For modelling, birth order is grouped to reduce sparsity in high-order births
while preserving meaningful differences in reproductive history.

Proposed categories:

- 1 = First birth
- 2–3 = Birth order 2–3
- 4–5 = Birth order 4–5
- 6+ = Birth order 6 or higher
- Unknown / missing = birth order not recorded

The original `bord_01` variable is retained alongside the grouped version for
reproducibility.

In [ ]:
# ======================================================================
# CONSTRUCT GROUPED BIRTH ORDER
#
# Raw variable:
#   bord_01 = birth order of the most recent live birth
#
# Derived categories:
#   First birth
#   Birth order 2-3
#   Birth order 4-5
#   Birth order 6+
#   Unknown / missing
#
# The original raw count remains unchanged.
# ======================================================================

birth_order_recode = birth_interval_recode.copy()

# ----------------------------------------------------------------------
# Initialize all observations as "Unknown / missing".
# ----------------------------------------------------------------------

birth_order_recode["birth_order_group"] = "Unknown / missing"

# ----------------------------------------------------------------------
# Assign interpretable birth-order groups.
# ----------------------------------------------------------------------

birth_order_recode.loc[
    birth_order_recode["bord_01"].eq(1),
    "birth_order_group"
] = "First birth"

birth_order_recode.loc[
    birth_order_recode["bord_01"].between(2, 3),
    "birth_order_group"
] = "Birth order 2-3"

birth_order_recode.loc[
    birth_order_recode["bord_01"].between(4, 5),
    "birth_order_group"
] = "Birth order 4-5"

birth_order_recode.loc[
    birth_order_recode["bord_01"].ge(6),
    "birth_order_group"
] = "Birth order 6+"


# ----------------------------------------------------------------------
# Define explicit category order.
# ----------------------------------------------------------------------

birth_order_categories = [
    "First birth",
    "Birth order 2-3",
    "Birth order 4-5",
    "Birth order 6+",
    "Unknown / missing"
]

birth_order_recode["birth_order_group"] = pd.Categorical(
    birth_order_recode["birth_order_group"],
    categories=birth_order_categories,
    ordered=False
)


# ----------------------------------------------------------------------
# Readable demonstration output
# ----------------------------------------------------------------------

print("\n" + "=" * 72)
print("bord_01 = Birth order of the most recent live birth")
print("Derived predictor = Grouped birth order")
print("=" * 72)

print("\nCategory definitions:")
print("  First birth       = bord_01 = 1")
print("  Birth order 2-3   = bord_01 = 2 or 3")
print("  Birth order 4-5   = bord_01 = 4 or 5")
print("  Birth order 6+    = bord_01 >= 6")
print("  Unknown / missing = bord_01 not recorded")

print("\nRaw birth-order range:")
print(
    "Minimum:",
    birth_order_recode["bord_01"].min(),
    "| Maximum:",
    birth_order_recode["bord_01"].max()
)

print("\nDerived distribution:")
print(
    birth_order_recode["birth_order_group"]
    .value_counts(dropna=False)
    .reindex(birth_order_categories)
)

print(
    "\nTotal:",
    len(birth_order_recode)
)

### 1.6.4 Geographic access to care

Two previously validated access measures are examined:

- `s1101`: travel time in minutes to the nearest hospital or health center;
- `s1101a`: usual transport mode to the nearest hospital or health center.

Travel time is retained initially as a continuous measure. The value `600`
represents a top-coded travel time of 600 minutes or more and is preserved as
such.

Zero-minute travel times are retained but explicitly audited because they may
represent residence at, adjacent to, or extremely close to a health facility.

Transport mode is initially retained in its original categories. Category
collapsing is considered only after examining the distribution in the ANC
analytic cohort.

In [ ]:
# ======================================================================
# AUDIT GEOGRAPHIC ACCESS VARIABLES
#
# Variables:
#   s1101  = Travel time in minutes to nearest hospital / health center
#   s1101a = Transport mode to nearest hospital / health center
#
# Purpose:
# Examine the raw distribution before deciding whether any transformation
# or category collapsing is needed.
#
# No recoding occurs in this cell.
# ======================================================================

access_df = birth_order_recode.copy()


# ----------------------------------------------------------------------
# s1101 = Travel time
# ----------------------------------------------------------------------

print("\n" + "=" * 76)
print("s1101 = Travel time to nearest hospital / health center")
print("Unit = minutes")
print("=" * 76)

print("\nSummary statistics:")
print(
    access_df["s1101"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nSpecial values of interest:")
print(
    "Zero-minute values:",
    access_df["s1101"].eq(0).sum()
)

print(
    "Top-coded values (600 minutes or more):",
    access_df["s1101"].eq(600).sum()
)

print(
    "Missing values:",
    access_df["s1101"].isna().sum()
)


# ----------------------------------------------------------------------
# s1101a = Transport mode
# ----------------------------------------------------------------------

print("\n" + "=" * 76)
print("s1101a = Transport mode to nearest hospital / health center")
print("=" * 76)

print("\nRaw category codes and frequencies:")

transport_counts = (
    access_df["s1101a"]
    .value_counts(dropna=False)
    .sort_index()
)

print(transport_counts)

print("\nNumber of observed transport categories:")
print(access_df["s1101a"].nunique(dropna=True))

print(
    "\nMissing transport-mode values:",
    access_df["s1101a"].isna().sum()
)

### 1.6.4 Geographic access: preprocessing decision

The previously validated preprocessing rules for geographic access are reused
because the source variables and coding structure are unchanged.

For `s1101`, travel time is retained as a continuous measure in minutes.
Reported values of 0 minutes are preserved rather than automatically treated
as errors. Values of 600 are retained as the EDHS top-coded category
representing 600 minutes or more. No trimming or winsorization is applied.

A supplementary grouped travel-time variable is also constructed:

- <30 minutes
- 30–59 minutes
- 60–119 minutes
- 120+ minutes

For `s1101a`, the original nine transport categories are grouped using the
previously validated mapping:

- Walking
- Motorcycle/bajaj/taxi/minibus
- Other motorized
- Other non-motorized

The grouped distributions are re-audited within the ANC analytic cohort.

In [ ]:
# ======================================================================
# PREPROCESS GEOGRAPHIC ACCESS VARIABLES
#
# Raw variables:
#   s1101  = travel time in minutes to nearest hospital / health center
#   s1101a = transport mode to nearest hospital / health center
#
# Previously validated mappings are reused, but frequencies are
# re-audited in the current ANC cohort.
# ======================================================================

access_recode = access_df.copy()


# ----------------------------------------------------------------------
# 1. Continuous travel time
# ----------------------------------------------------------------------

access_recode["travel_time_min"] = access_recode["s1101"].copy()


# ----------------------------------------------------------------------
# 2. Supplementary grouped travel time
#
# Categories:
#   <30 min
#   30-59 min
#   60-119 min
#   120+ min
# ----------------------------------------------------------------------

access_recode["travel_time_group"] = pd.cut(
    access_recode["s1101"],
    bins=[-np.inf, 29, 59, 119, np.inf],
    labels=[
        "<30 min",
        "30-59 min",
        "60-119 min",
        "120+ min"
    ],
    right=True
)


# ----------------------------------------------------------------------
# 3. Group transport mode
#
# Raw s1101a coding used in the validated preprocessing register:
#
#   24             = Walking
#   13             = Motorcycle / bajaj / taxi / minibus
#   11, 12, 14     = Other motorized
#   21, 22, 23, 25 = Other non-motorized
# ----------------------------------------------------------------------

transport_map = {
    24: "Walking",
    13: "Motorcycle/bajaj/taxi/minibus",

    11: "Other motorized",
    12: "Other motorized",
    14: "Other motorized",

    21: "Other non-motorized",
    22: "Other non-motorized",
    23: "Other non-motorized",
    25: "Other non-motorized"
}

access_recode["transport_mode_group"] = (
    access_recode["s1101a"]
    .map(transport_map)
)


# ----------------------------------------------------------------------
# Explicit category order
# ----------------------------------------------------------------------

transport_categories = [
    "Walking",
    "Motorcycle/bajaj/taxi/minibus",
    "Other motorized",
    "Other non-motorized"
]

access_recode["transport_mode_group"] = pd.Categorical(
    access_recode["transport_mode_group"],
    categories=transport_categories
)


# ----------------------------------------------------------------------
# Demonstration output
# ----------------------------------------------------------------------

print("\n" + "=" * 76)
print("s1101 = Travel time to nearest hospital / health center")
print("Derived predictors = travel_time_min + travel_time_group")
print("=" * 76)

print("\nContinuous travel time:")
print(access_recode["travel_time_min"].describe())

print("\nGrouped travel-time definitions:")
print("  <30 min      = 0-29 minutes")
print("  30-59 min    = 30-59 minutes")
print("  60-119 min   = 60-119 minutes")
print("  120+ min     = 120 minutes or more")

print("\nGrouped travel-time distribution:")
print(
    access_recode["travel_time_group"]
    .value_counts()
    .sort_index()
)


print("\n" + "=" * 76)
print("s1101a = Transport mode to nearest hospital / health center")
print("Derived predictor = transport_mode_group")
print("=" * 76)

print("\nCategory definitions:")
print("  Walking                         = code 24")
print("  Motorcycle/bajaj/taxi/minibus  = code 13")
print("  Other motorized                 = codes 11, 12, 14")
print("  Other non-motorized             = codes 21, 22, 23, 25")

print("\nGrouped transport-mode distribution:")
print(
    access_recode["transport_mode_group"]
    .value_counts(dropna=False)
    .reindex(transport_categories)
)

print(
    "\nUnmapped / missing transport values:",
    access_recode["transport_mode_group"].isna().sum()
)

### 1.6.5 Pregnancy intention

`m10_1` captures the intention status of the index pregnancy.

Because pregnancy intention conceptually precedes subsequent ANC utilization,
it is retained as a primary candidate predictor.

The raw category coding is examined before any modelling representation is
finalized.

In [ ]:
# ======================================================================
# AUDIT PREGNANCY INTENTION
#
# Variable:
#   m10_1 = Pregnancy intention for the index pregnancy
#
# Purpose:
# Display the raw category distribution with readable category labels.
#
# No collapsing is performed in this cell.
# ======================================================================

pregnancy_intention_df = access_recode.copy()

print("\n" + "=" * 76)
print("m10_1 = Pregnancy intention")
print("=" * 76)

print("\nRaw codes and frequencies:")
print(
    pregnancy_intention_df["m10_1"]
    .value_counts(dropna=False)
    .sort_index()
)

# ----------------------------------------------------------------------
# Readable category labels
# ----------------------------------------------------------------------

pregnancy_intention_labels = {
    1: "Wanted then",
    2: "Wanted later",
    3: "Wanted no more"
}

pregnancy_intention_df["pregnancy_intention"] = (
    pregnancy_intention_df["m10_1"]
    .map(pregnancy_intention_labels)
)

print("\nCategory definitions:")
print("  1 = Wanted then")
print("  2 = Wanted later")
print("  3 = Wanted no more")

print("\nLabelled distribution:")
print(
    pregnancy_intention_df["pregnancy_intention"]
    .value_counts(dropna=False)
)

print(
    "\nUnmapped / missing values:",
    pregnancy_intention_df["pregnancy_intention"].isna().sum()
)

### 1.6.6 Maternal age

For the ANC8 prediction study, maternal age is represented by `v012`, the
respondent's age at interview.

The previously derived age at index birth was appropriate for the earlier
birth-specific transition-safety analysis, but it is not adopted as the
primary age predictor in this study. The ANC8 analysis is not restricted to
facility births and is framed around maternal characteristics measured in the
EDHS interview.

Accordingly:

- `v012` is retained as the primary maternal-age predictor;
- age at index birth is not used in the primary ANC8 model;
- the earlier derivation remains documented only as a validation/exploratory
  construct if later needed.

In [ ]:
# ======================================================================
# AUDIT MATERNAL AGE AT INTERVIEW
#
# Variable:
#   v012 = respondent's age at interview, in completed years
#
# This is the primary maternal-age predictor for the ANC8 study.
# ======================================================================

maternal_age_df = pregnancy_intention_df.merge(
    ir_df[
        ["caseid", "v012"]
    ],
    on="caseid",
    how="left",
    validate="one_to_one"
)

print("\n" + "=" * 76)
print("v012 = Maternal age at interview")
print("=" * 76)

print("\nSummary statistics:")
print(
    maternal_age_df["v012"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

print("\nMissing values:")
print(
    maternal_age_df["v012"].isna().sum()
)

print("\nAge distribution by conventional groups:")
age_groups = pd.cut(
    maternal_age_df["v012"],
    bins=[14, 19, 24, 29, 34, 39, 49],
    labels=[
        "15-19",
        "20-24",
        "25-29",
        "30-34",
        "35-39",
        "40-49"
    ],
    include_lowest=True
)

print(
    age_groups
    .value_counts()
    .sort_index()
)

### 1.6.7 Maternal education

Maternal education is measured using `v106`, the highest educational level
attained by the respondent.

The variable is complete in the ANC analytic cohort and is retained initially
using its original EDHS categories. No category collapsing is performed unless
later justified by sparse cells or modelling considerations.

In [ ]:
# ======================================================================
# AUDIT MATERNAL EDUCATION
#
# Variable:
#   v106 = Highest educational level attained by the respondent
#
# Purpose:
# Display the original EDHS categories and frequencies in the ANC cohort.
#
# No collapsing is performed in this cell.
# ======================================================================

education_df = maternal_age_df.copy()

education_labels = {
    0: "No education",
    1: "Primary",
    2: "Secondary",
    3: "Higher"
}

education_df["maternal_education"] = (
    education_df["v106"]
    .map(education_labels)
)

print("\n" + "=" * 76)
print("v106 = Maternal education")
print("=" * 76)

print("\nCategory definitions:")
print("  0 = No education")
print("  1 = Primary")
print("  2 = Secondary")
print("  3 = Higher")

print("\nRaw codes and frequencies:")
print(
    education_df["v106"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nLabelled distribution:")
print(
    education_df["maternal_education"]
    .value_counts(dropna=False)
)

print(
    "\nUnmapped / missing values:",
    education_df["maternal_education"].isna().sum()
)

### 1.6.8 Household wealth, residence, and region

Household wealth, place of residence, and administrative region are retained as
primary contextual predictors because they capture socioeconomic and geographic
differences in access to antenatal care.

The original EDHS categories are retained at this stage. No collapsing is
performed before examining their distribution in the ANC analytic cohort.

In [ ]:
# ======================================================================
# AUDIT WEALTH, RESIDENCE, AND REGION
#
# Variables:
#   v190 = Household wealth quintile
#   v025 = Place of residence
#   v024 = Region
#
# Purpose:
# Display readable category labels and frequencies in the ANC cohort.
# ======================================================================

context_df = education_df.copy()

# ----------------------------------------------------------------------
# Wealth quintile
# ----------------------------------------------------------------------

wealth_labels = {
    1: "Poorest",
    2: "Poorer",
    3: "Middle",
    4: "Richer",
    5: "Richest"
}

context_df["wealth_quintile"] = (
    context_df["v190"]
    .map(wealth_labels)
)


# ----------------------------------------------------------------------
# Residence
# ----------------------------------------------------------------------

residence_labels = {
    1: "Urban",
    2: "Rural"
}

context_df["residence"] = (
    context_df["v025"]
    .map(residence_labels)
)


# ----------------------------------------------------------------------
# Region
# Keep raw codes for now.
# We will attach exact EDHS region names after confirming the coding.
# ----------------------------------------------------------------------

print("\n" + "=" * 76)
print("v190 = Household wealth quintile")
print("=" * 76)

print("\nCategory definitions:")
print("  1 = Poorest")
print("  2 = Poorer")
print("  3 = Middle")
print("  4 = Richer")
print("  5 = Richest")

print("\nDistribution:")
print(
    context_df["wealth_quintile"]
    .value_counts(dropna=False)
    .reindex([
        "Poorest",
        "Poorer",
        "Middle",
        "Richer",
        "Richest"
    ])
)


print("\n" + "=" * 76)
print("v025 = Place of residence")
print("=" * 76)

print("\nCategory definitions:")
print("  1 = Urban")
print("  2 = Rural")

print("\nDistribution:")
print(
    context_df["residence"]
    .value_counts(dropna=False)
    .reindex(["Urban", "Rural"])
)


print("\n" + "=" * 76)
print("v024 = Region")
print("=" * 76)

print("\nRaw region codes and frequencies:")
print(
    context_df["v024"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "\nNumber of observed regions:",
    context_df["v024"].nunique(dropna=True)
)

### 1.6.8 Region: preprocessing decision

Region is represented by `v024`.

The previously fixed preprocessing register verified the original Stata value
labels directly from the EDHS source dataset and confirmed that `v024` is
equivalent to `v101`.

All 14 regional categories are retained separately because the study has a
national coverage and representativeness focus. No regional collapsing is
performed at this stage.

In [ ]:
# ======================================================================
# LABEL REGION
#
# Variable:
#   v024 = Region
#
# The mapping below reuses the previously validated EDHS Stata value
# labels from the fixed preprocessing register.
# ======================================================================

region_labels = {
    1:  "Tigray",
    2:  "Afar",
    3:  "Amhara",
    4:  "Oromia",
    5:  "Somali",
    6:  "Benishangul-Gumuz",
    7:  "Central Ethiopia",
    8:  "Sidama",
    9:  "South West Ethiopia",
    10: "South Ethiopia",
    12: "Gambella",
    13: "Harari",
    14: "Addis Ababa",
    15: "Dire Dawa"
}

context_df["region"] = context_df["v024"].map(region_labels)

print("\n" + "=" * 76)
print("v024 = Region")
print("Derived predictor = region")
print("=" * 76)

print("\nCategory definitions:")
for code, label in region_labels.items():
    print(f"  {code:>2} = {label}")

print("\nRegional distribution:")
print(
    context_df["region"]
    .value_counts(dropna=False)
    .reindex(region_labels.values())
)

print(
    "\nUnmapped / missing values:",
    context_df["region"].isna().sum()
)

print(
    "Number of labelled regions:",
    context_df["region"].nunique(dropna=True)
)

### 1.6.9 Household composition, religion, media exposure, internet use, and health insurance

Several background characteristics are complete in the ANC analytic cohort and
are retained as primary candidate predictors:

- household size (`v136`);
- religion (`v130`);
- newspaper exposure (`v157`);
- radio exposure (`v158`);
- television exposure (`v159`);
- internet exposure (`v171a`);
- health insurance coverage (`v481`).

Original categories are preserved at this stage. Household size is retained as
a continuous count variable, while the remaining variables are represented as
categorical predictors using their validated EDHS coding.

In [ ]:
# ======================================================================
# AUDIT HOUSEHOLD, RELIGION, MEDIA, INTERNET, AND INSURANCE VARIABLES
#
# Purpose:
# Attach readable labels to complete background predictors and verify
# that all observed raw codes are mapped.
# ======================================================================

background_df = context_df.copy()


# ----------------------------------------------------------------------
# v136 = Household size
# ----------------------------------------------------------------------

background_df["household_size"] = background_df["v136"].copy()

print("\n" + "=" * 76)
print("v136 = Household size")
print("Derived predictor = household_size")
print("=" * 76)

print("\nSummary statistics:")
print(
    background_df["household_size"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print(
    "\nMissing values:",
    background_df["household_size"].isna().sum()
)


# ----------------------------------------------------------------------
# v130 = Religion
#
# IMPORTANT:
# First display raw codes.
# We will attach exact category names only from the validated register /
# source coding, rather than guess them.
# ----------------------------------------------------------------------

print("\n" + "=" * 76)
print("v130 = Religion")
print("=" * 76)

print("\nRaw codes and frequencies:")
print(
    background_df["v130"]
    .value_counts(dropna=False)
    .sort_index()
)


# ----------------------------------------------------------------------
# Media exposure variables
#
# v157 = frequency of reading newspaper / magazine
# v158 = frequency of listening to radio
# v159 = frequency of watching television
#
# Standard EDHS frequency coding retained:
#   0 = Not at all
#   1 = Less than once a week
#   2 = At least once a week
# ----------------------------------------------------------------------

media_labels = {
    0: "Not at all",
    1: "Less than once a week",
    2: "At least once a week"
}

media_vars = {
    "v157": "Newspaper / magazine exposure",
    "v158": "Radio exposure",
    "v159": "Television exposure"
}

for var, meaning in media_vars.items():

    derived_name = {
        "v157": "newspaper_exposure",
        "v158": "radio_exposure",
        "v159": "television_exposure"
    }[var]

    background_df[derived_name] = background_df[var].map(media_labels)

    print("\n" + "=" * 76)
    print(f"{var} = {meaning}")
    print("=" * 76)

    print("\nCategory definitions:")
    print("  0 = Not at all")
    print("  1 = Less than once a week")
    print("  2 = At least once a week")

    print("\nLabelled distribution:")
    print(
        background_df[derived_name]
        .value_counts(dropna=False)
    )

    print(
        "Unmapped / missing values:",
        background_df[derived_name].isna().sum()
    )


# ----------------------------------------------------------------------
# v171a = Internet use
#
# Display raw codes first and attach labels only after confirming the
# exact validated coding.
# ----------------------------------------------------------------------

print("\n" + "=" * 76)
print("v171a = Internet exposure / use")
print("=" * 76)

print("\nRaw codes and frequencies:")
print(
    background_df["v171a"]
    .value_counts(dropna=False)
    .sort_index()
)


# ----------------------------------------------------------------------
# v481 = Health insurance coverage
# ----------------------------------------------------------------------

insurance_labels = {
    0: "No",
    1: "Yes"
}

background_df["health_insurance"] = (
    background_df["v481"]
    .map(insurance_labels)
)

print("\n" + "=" * 76)
print("v481 = Health insurance coverage")
print("=" * 76)

print("\nCategory definitions:")
print("  0 = No")
print("  1 = Yes")

print("\nLabelled distribution:")
print(
    background_df["health_insurance"]
    .value_counts(dropna=False)
)

print(
    "Unmapped / missing values:",
    background_df["health_insurance"].isna().sum()
)

### 1.6.9 Background information-access and religion variables: preprocessing decision

Previously validated preprocessing rules are reused for religion and internet
exposure.

Religion (`v130`) is grouped into four analytically meaningful categories:

- Orthodox
- Protestant
- Muslim
- Other religion

The `Other religion` category combines the relatively sparse Catholic,
Traditional, and original Other categories.

Internet exposure (`v171a`) is retained using its validated timing categories:

- Never used
- Used in the last 12 months
- Used before the last 12 months

Household size remains a continuous count variable. Newspaper, radio, and
television exposure retain their original three frequency categories, and
health insurance remains binary.

In [ ]:
# ======================================================================
# FINALIZE RELIGION AND INTERNET-EXPOSURE VARIABLES
#
# Purpose:
# Reuse the previously validated preprocessing rules from the fixed
# predictor preprocessing register and re-audit the distributions in the
# current ANC cohort.
# ======================================================================

background_final = background_df.copy()


# ----------------------------------------------------------------------
# v130 = Religion
#
# Raw EDHS coding:
#   1  = Orthodox
#   2  = Catholic
#   3  = Protestant
#   4  = Muslim
#   5  = Traditional
#   96 = Other
#
# Final analytic grouping:
#   Orthodox
#   Protestant
#   Muslim
#   Other religion
# ----------------------------------------------------------------------

religion_map = {
    1: "Orthodox",
    2: "Other religion",   # Catholic
    3: "Protestant",
    4: "Muslim",
    5: "Other religion",   # Traditional
    96: "Other religion"   # Other
}

background_final["religion_group"] = (
    background_final["v130"]
    .map(religion_map)
)

religion_categories = [
    "Orthodox",
    "Protestant",
    "Muslim",
    "Other religion"
]

background_final["religion_group"] = pd.Categorical(
    background_final["religion_group"],
    categories=religion_categories
)


print("\n" + "=" * 76)
print("v130 = Religion")
print("Derived predictor = religion_group")
print("=" * 76)

print("\nRaw category definitions:")
print("   1 = Orthodox")
print("   2 = Catholic")
print("   3 = Protestant")
print("   4 = Muslim")
print("   5 = Traditional")
print("  96 = Other")

print("\nFinal analytic grouping:")
print("  Orthodox       = raw code 1")
print("  Protestant     = raw code 3")
print("  Muslim         = raw code 4")
print("  Other religion = raw codes 2, 5, 96")

print("\nGrouped distribution:")
print(
    background_final["religion_group"]
    .value_counts(dropna=False)
    .reindex(religion_categories)
)

print(
    "\nUnmapped / missing religion values:",
    background_final["religion_group"].isna().sum()
)


# ----------------------------------------------------------------------
# v171a = Internet exposure
#
# Validated EDHS categories:
#   0 = Never used
#   1 = Used in last 12 months
#   2 = Used before last 12 months
# ----------------------------------------------------------------------

internet_map = {
    0: "Never used",
    1: "Used in last 12 months",
    2: "Used before last 12 months"
}

background_final["internet_exposure"] = (
    background_final["v171a"]
    .map(internet_map)
)

internet_categories = [
    "Never used",
    "Used in last 12 months",
    "Used before last 12 months"
]

background_final["internet_exposure"] = pd.Categorical(
    background_final["internet_exposure"],
    categories=internet_categories
)


print("\n" + "=" * 76)
print("v171a = Internet exposure / use")
print("Derived predictor = internet_exposure")
print("=" * 76)

print("\nCategory definitions:")
print("  0 = Never used")
print("  1 = Used in last 12 months")
print("  2 = Used before last 12 months")

print("\nLabelled distribution:")
print(
    background_final["internet_exposure"]
    .value_counts(dropna=False)
    .reindex(internet_categories)
)

print(
    "\nUnmapped / missing internet values:",
    background_final["internet_exposure"].isna().sum()
)

### 1.6.10 Marital status, partner characteristics, and healthcare decision-making

Marital/union status and partner-related variables are retained because they
may influence women's ability to initiate and sustain antenatal care.

Partner education (`v701`), partner occupation (`v705`), and healthcare
decision-making (`v743a`) are structurally not applicable to women who are not
currently in union.

The previously identified structural non-applicability is therefore preserved
explicitly. Original substantive categories are retained where feasible, and
sparse categories are reviewed before any collapsing.

In [ ]:
# ======================================================================
# AUDIT AND LABEL MARITAL / PARTNER / AUTONOMY VARIABLES
#
# Variables:
#   v501  = Marital / union status
#   v701  = Partner education
#   v705  = Partner occupation
#   v743a = Person who usually decides on respondent's healthcare
#
# Purpose:
# Attach readable labels while preserving structural non-applicability.
# ======================================================================

relationship_df = background_final.copy()


# ----------------------------------------------------------------------
# v501 = Marital / union status
# ----------------------------------------------------------------------

marital_labels = {
    0: "Never in union",
    1: "Married",
    2: "Living with partner",
    3: "Widowed",
    4: "Divorced",
    5: "Separated"
}

relationship_df["marital_status"] = (
    relationship_df["v501"]
    .map(marital_labels)
)

print("\n" + "=" * 76)
print("v501 = Marital / union status")
print("=" * 76)

print("\nCategory definitions:")
for code, label in marital_labels.items():
    print(f"  {code} = {label}")

print("\nLabelled distribution:")
print(
    relationship_df["marital_status"]
    .value_counts(dropna=False)
)


# ----------------------------------------------------------------------
# v701 = Partner education
#
# Structural non-applicability is already known to occur among women
# not currently in union.
# ----------------------------------------------------------------------

partner_education_labels = {
    0: "No education",
    1: "Primary",
    2: "Secondary",
    3: "Higher",
    8: "Don't know"
}

relationship_df["partner_education"] = (
    relationship_df["v701"]
    .map(partner_education_labels)
)

relationship_df.loc[
    relationship_df["v501"].isin([0, 3, 4, 5]) &
    relationship_df["v701"].isna(),
    "partner_education"
] = "Not applicable / not currently in union"


print("\n" + "=" * 76)
print("v701 = Partner education")
print("=" * 76)

print("\nCategory definitions:")
print("  0 = No education")
print("  1 = Primary")
print("  2 = Secondary")
print("  3 = Higher")
print("  8 = Don't know")
print("  NA outside current union = Not applicable / not currently in union")

print("\nLabelled distribution:")
print(
    relationship_df["partner_education"]
    .value_counts(dropna=False)
)


# ----------------------------------------------------------------------
# v705 = Partner occupation
#
# Keep raw category codes visible first.
# We will finalize grouping only after reviewing the category sizes.
# ----------------------------------------------------------------------

print("\n" + "=" * 76)
print("v705 = Partner occupation")
print("=" * 76)

print("\nRaw codes and frequencies:")
print(
    relationship_df["v705"]
    .value_counts(dropna=False)
    .sort_index()
)


# ----------------------------------------------------------------------
# v743a = Healthcare decision-making
#
# Raw coding:
#   1 = Respondent alone
#   2 = Respondent and husband/partner jointly
#   4 = Husband/partner alone
#   5 = Someone else
#
# Missing outside current union is structural.
# One remaining applicable missing value is retained as Unknown/missing.
# ----------------------------------------------------------------------

decision_labels = {
    1: "Respondent alone",
    2: "Joint with husband/partner",
    4: "Husband/partner alone",
    5: "Someone else"
}

relationship_df["healthcare_decision_making"] = (
    relationship_df["v743a"]
    .map(decision_labels)
)

relationship_df.loc[
    relationship_df["v501"].isin([0, 3, 4, 5]) &
    relationship_df["v743a"].isna(),
    "healthcare_decision_making"
] = "Not applicable / not currently in union"

relationship_df.loc[
    relationship_df["v501"].isin([1, 2]) &
    relationship_df["v743a"].isna(),
    "healthcare_decision_making"
] = "Unknown / missing"


print("\n" + "=" * 76)
print("v743a = Healthcare decision-making")
print("=" * 76)

print("\nCategory definitions:")
print("  1 = Respondent alone")
print("  2 = Joint with husband/partner")
print("  4 = Husband/partner alone")
print("  5 = Someone else")
print("  NA outside current union = Not applicable / not currently in union")

print("\nLabelled distribution:")
print(
    relationship_df["healthcare_decision_making"]
    .value_counts(dropna=False)
)

### 1.6.10 Partner occupation and healthcare decision-making: final preprocessing

Previously validated preprocessing decisions are reused for partner occupation
and healthcare decision-making.

Partner occupation (`v705`) is collapsed from the original detailed occupation
codes into broader occupational groups to avoid sparse categories while
preserving substantive distinctions.

Healthcare decision-making is represented primarily as participation in the
respondent's own healthcare decision:

- Participates: respondent decides alone or jointly with husband/partner
- Does not participate: husband/partner or someone else decides
- Not applicable: respondent is not currently in union
- Unknown/missing: applicable decision information is unavailable

Structural non-applicability remains distinct from ordinary missingness.

In [ ]:
# ======================================================================
# FINALIZE PARTNER OCCUPATION AND HEALTHCARE DECISION-MAKING
#
# Variables:
#   v705  = Partner occupation
#   v743a = Person who usually decides on respondent's healthcare
#
# Previously validated groupings are reused and re-audited in the
# current ANC cohort.
# ======================================================================

relationship_final = relationship_df.copy()


# ----------------------------------------------------------------------
# 1. Partner occupation
#
# Raw v705 coding:
#   0  = Did not work
#   1  = Professional / technical / managerial
#   2  = Clerical
#   3  = Sales
#   4  = Agricultural self-employed
#   5  = Agricultural employee
#   6  = Household / domestic
#   7  = Services
#   8  = Skilled manual
#   9  = Unskilled manual
#   98 = Don't know
#
# Final analytic grouping:
#   Did not work
#   Professional / clerical
#   Sales / services
#   Agriculture
#   Manual / domestic
#   Unknown
#   Not applicable / not currently in union
# ----------------------------------------------------------------------

partner_occupation_map = {
    0: "Did not work",

    1: "Professional / clerical",
    2: "Professional / clerical",

    3: "Sales / services",
    7: "Sales / services",

    4: "Agriculture",
    5: "Agriculture",

    6: "Manual / domestic",
    8: "Manual / domestic",
    9: "Manual / domestic",

    98: "Unknown"
}

relationship_final["partner_occupation_group"] = (
    relationship_final["v705"]
    .map(partner_occupation_map)
)

# Preserve structural non-applicability explicitly.
relationship_final.loc[
    relationship_final["v501"].isin([0, 3, 4, 5]) &
    relationship_final["v705"].isna(),
    "partner_occupation_group"
] = "Not applicable / not currently in union"


partner_occupation_categories = [
    "Did not work",
    "Professional / clerical",
    "Sales / services",
    "Agriculture",
    "Manual / domestic",
    "Unknown",
    "Not applicable / not currently in union"
]

relationship_final["partner_occupation_group"] = pd.Categorical(
    relationship_final["partner_occupation_group"],
    categories=partner_occupation_categories
)


print("\n" + "=" * 76)
print("v705 = Partner occupation")
print("Derived predictor = partner_occupation_group")
print("=" * 76)

print("\nRaw category definitions:")
print("   0 = Did not work")
print("   1 = Professional / technical / managerial")
print("   2 = Clerical")
print("   3 = Sales")
print("   4 = Agricultural self-employed")
print("   5 = Agricultural employee")
print("   6 = Household / domestic")
print("   7 = Services")
print("   8 = Skilled manual")
print("   9 = Unskilled manual")
print("  98 = Don't know")

print("\nFinal analytic grouping:")
print("  Did not work             = 0")
print("  Professional / clerical  = 1, 2")
print("  Sales / services         = 3, 7")
print("  Agriculture              = 4, 5")
print("  Manual / domestic        = 6, 8, 9")
print("  Unknown                  = 98")
print("  Not applicable           = not currently in union")

print("\nGrouped distribution:")
print(
    relationship_final["partner_occupation_group"]
    .value_counts(dropna=False)
    .reindex(partner_occupation_categories)
)

print(
    "\nUnmapped / missing values:",
    relationship_final["partner_occupation_group"].isna().sum()
)


# ----------------------------------------------------------------------
# 2. Healthcare decision-making participation
#
# v743a raw coding observed in this cohort:
#   1 = Respondent alone
#   2 = Respondent and husband/partner jointly
#   4 = Husband/partner alone
#   5 = Someone else
#
# Final primary construct:
#   Participates          = codes 1 or 2
#   Does not participate = codes 4 or 5
#   Not applicable       = not currently in union
#   Unknown / missing    = applicable but source value missing
# ----------------------------------------------------------------------

relationship_final["healthcare_decision_participation"] = np.nan
relationship_final[
    "healthcare_decision_participation"
] = relationship_final[
    "healthcare_decision_participation"
].astype("object")

relationship_final.loc[
    relationship_final["v743a"].isin([1, 2]),
    "healthcare_decision_participation"
] = "Participates"

relationship_final.loc[
    relationship_final["v743a"].isin([4, 5]),
    "healthcare_decision_participation"
] = "Does not participate"

relationship_final.loc[
    relationship_final["v501"].isin([0, 3, 4, 5]) &
    relationship_final["v743a"].isna(),
    "healthcare_decision_participation"
] = "Not applicable / not currently in union"

relationship_final.loc[
    relationship_final["v501"].isin([1, 2]) &
    relationship_final["v743a"].isna(),
    "healthcare_decision_participation"
] = "Unknown / missing"


decision_categories = [
    "Participates",
    "Does not participate",
    "Not applicable / not currently in union",
    "Unknown / missing"
]

relationship_final["healthcare_decision_participation"] = pd.Categorical(
    relationship_final["healthcare_decision_participation"],
    categories=decision_categories
)


print("\n" + "=" * 76)
print("v743a = Decision on respondent's healthcare")
print("Derived predictor = healthcare_decision_participation")
print("=" * 76)

print("\nCategory definitions:")
print("  Participates          = respondent alone or jointly (codes 1, 2)")
print("  Does not participate  = husband/partner or someone else (codes 4, 5)")
print("  Not applicable        = respondent not currently in union")
print("  Unknown / missing     = applicable source value missing")

print("\nDerived distribution:")
print(
    relationship_final["healthcare_decision_participation"]
    .value_counts(dropna=False)
    .reindex(decision_categories)
)

print(
    "\nUnmapped / missing values:",
    relationship_final[
        "healthcare_decision_participation"
    ].isna().sum()
)

### 1.6.11 Previous adverse pregnancy outcome

Previous pregnancy history may influence subsequent ANC utilization and is
therefore considered a primary candidate predictor.

This construct cannot be derived by simply inspecting pregnancy-history
position 01 because that position does not necessarily correspond to the index
live birth.

The validated mapping therefore uses:

- `pidxb_01` to `pidxb_20` to identify the pregnancy-history position linked
  to the index live birth;
- `p32_01` to `p32_20` to identify outcomes of pregnancies occurring before
  that index pregnancy.

The intended derived predictor distinguishes:

- no previous pregnancy;
- previous pregnancy but no adverse outcome;
- at least one previous adverse pregnancy outcome.

Before constructing this predictor, the linkage and raw outcome coding are
audited in the current ANC analytic cohort.

In [ ]:
# ======================================================================
# AUDIT PREGNANCY-HISTORY LINKAGE FOR PREVIOUS ADVERSE OUTCOME
#
# Source variables:
#   pidxb_01 ... pidxb_20 = linkage of pregnancy-history positions to
#                            birth-history records
#   p32_01   ... p32_20   = pregnancy outcome
#
# Purpose:
# Verify that the required pregnancy-history variables are available and
# inspect their raw coding before deriving a previous-adverse-outcome
# predictor.
#
# IMPORTANT:
# No previous-adverse-outcome variable is constructed in this cell.
# ======================================================================

previous_outcome_df = relationship_final.copy()

# ----------------------------------------------------------------------
# Define pregnancy-history source-variable names.
# ----------------------------------------------------------------------

pidxb_vars = [f"pidxb_{i:02d}" for i in range(1, 21)]
p32_vars   = [f"p32_{i:02d}"   for i in range(1, 21)]

pregnancy_history_vars = pidxb_vars + p32_vars


# ----------------------------------------------------------------------
# Check availability in the original IR dataset.
# ----------------------------------------------------------------------

missing_history_vars = [
    var for var in pregnancy_history_vars
    if var not in ir_df.columns
]

print("\n" + "=" * 78)
print("Pregnancy-history source-variable availability")
print("=" * 78)

print(
    "Expected variables:",
    len(pregnancy_history_vars)
)

print(
    "Variables missing from IR dataset:",
    missing_history_vars
)


# ----------------------------------------------------------------------
# Attach pregnancy-history variables to the ANC cohort.
# ----------------------------------------------------------------------

previous_outcome_df = previous_outcome_df.merge(
    ir_df[
        ["caseid"] + pregnancy_history_vars
    ],
    on="caseid",
    how="left",
    validate="one_to_one"
)


# ----------------------------------------------------------------------
# Count how many pregnancy-history positions are populated per woman.
# ----------------------------------------------------------------------

previous_outcome_df["n_pregnancy_history_records"] = (
    previous_outcome_df[p32_vars]
    .notna()
    .sum(axis=1)
)

print("\n" + "=" * 78)
print("Number of recorded pregnancy-history positions per woman")
print("=" * 78)

print(
    previous_outcome_df["n_pregnancy_history_records"]
    .value_counts(dropna=False)
    .sort_index()
)


# ----------------------------------------------------------------------
# Inspect raw p32 outcome codes across ALL pregnancy-history positions.
#
# This pools positions only for coding verification. It does NOT yet
# determine which pregnancies occurred before the index live birth.
# ----------------------------------------------------------------------

p32_long = (
    previous_outcome_df[p32_vars]
    .stack()
)

print("\n" + "=" * 78)
print("p32_01-p32_20 = Pregnancy outcome")
print("Pooled raw codes across recorded pregnancy-history positions")
print("=" * 78)

print(
    p32_long
    .value_counts(dropna=False)
    .sort_index()
)


# ----------------------------------------------------------------------
# Inspect pidxb linkage values.
#
# These values identify linkage between pregnancy-history records and
# birth-history records. We inspect them before applying the linkage rule.
# ----------------------------------------------------------------------

pidxb_long = (
    previous_outcome_df[pidxb_vars]
    .stack()
)

print("\n" + "=" * 78)
print("pidxb_01-pidxb_20 = Pregnancy-to-birth linkage variables")
print("Pooled observed values")
print("=" * 78)

print(
    pidxb_long
    .value_counts(dropna=False)
    .sort_index()
)

#### 1.6.11.1 Validation of the index-pregnancy linkage

The pregnancy-history position corresponding to the most recent live birth is
identified using the `pidxb_*` variables.

For the most recent live birth, the linked pregnancy-history record should have
`pidxb = 1`, because birth-history index 1 corresponds to the most recent live
birth.

Before defining prior pregnancies, the linkage is validated by confirming:

- that women with a recorded most recent live birth have exactly one
  pregnancy-history record linked to birth-history index 1; and
- that the outcome of this linked pregnancy is coded as a live birth.

Only after this validation are pregnancy-history positions occurring before the
index pregnancy used to construct previous adverse pregnancy outcome.

In [ ]:
# ======================================================================
# VALIDATE LINKAGE TO THE MOST RECENT LIVE BIRTH
#
# Key principle:
#   pidxb_* = 1 identifies the pregnancy-history record linked to
#   birth-history index 1, i.e. the woman's most recent live birth.
#
# We first identify that pregnancy-history position for each woman and
# then inspect the corresponding p32 pregnancy outcome.
#
# No previous-adverse-outcome predictor is constructed yet.
# ======================================================================

linkage_validation_df = previous_outcome_df.copy()

# ----------------------------------------------------------------------
# For each pregnancy-history position, flag whether it links to the
# most recent live birth (pidxb = 1).
# ----------------------------------------------------------------------

index_link_flags = pd.DataFrame(
    {
        f"index_link_{i:02d}":
            linkage_validation_df[f"pidxb_{i:02d}"].eq(1)
        for i in range(1, 21)
    },
    index=linkage_validation_df.index
)

# Number of pregnancy-history records linked to birth-history index 1.
linkage_validation_df["n_index_live_birth_links"] = (
    index_link_flags.sum(axis=1)
)

print("\n" + "=" * 78)
print("Validation of pregnancy-history linkage to most recent live birth")
print("=" * 78)

print("\nNumber of pidxb_* records equal to 1 per woman:")
print(
    linkage_validation_df["n_index_live_birth_links"]
    .value_counts(dropna=False)
    .sort_index()
)


# ----------------------------------------------------------------------
# Identify the pregnancy-history POSITION containing the index live birth.
#
# Example:
#   if pidxb_01 = 1 -> index pregnancy is pregnancy-history position 1
#   if pidxb_02 = 1 -> index pregnancy is position 2
#   etc.
# ----------------------------------------------------------------------

def get_index_pregnancy_position(row):
    """
    Return the pregnancy-history position (1-20) linked to the
    most recent live birth.

    Returns NaN if no unique pidxb value of 1 is found.
    """
    matches = [
        i
        for i in range(1, 21)
        if row[f"pidxb_{i:02d}"] == 1
    ]

    if len(matches) == 1:
        return matches[0]

    return np.nan


linkage_validation_df["index_pregnancy_position"] = (
    linkage_validation_df.apply(
        get_index_pregnancy_position,
        axis=1
    )
)


print("\n" + "=" * 78)
print("Pregnancy-history position of the most recent live birth")
print("=" * 78)

print(
    linkage_validation_df["index_pregnancy_position"]
    .value_counts(dropna=False)
    .sort_index()
)


# ----------------------------------------------------------------------
# Retrieve p32 for the linked index pregnancy.
# ----------------------------------------------------------------------

def get_index_pregnancy_outcome(row):
    """
    Retrieve p32 outcome code from the pregnancy-history position
    identified as the most recent live birth.
    """
    pos = row["index_pregnancy_position"]

    if pd.isna(pos):
        return np.nan

    return row[f"p32_{int(pos):02d}"]


linkage_validation_df["index_pregnancy_p32"] = (
    linkage_validation_df.apply(
        get_index_pregnancy_outcome,
        axis=1
    )
)


print("\n" + "=" * 78)
print("p32 outcome code of the linked most recent live birth")
print("=" * 78)

print(
    linkage_validation_df["index_pregnancy_p32"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nExpected interpretation:")
print("  The linked index pregnancy should consistently carry the")
print("  p32 code corresponding to a live-born outcome.")

print(
    "\nWomen without exactly one valid index linkage:",
    linkage_validation_df["index_pregnancy_position"].isna().sum()
)

#### 1.6.11.2 Audit of maternal-health records without an index live-birth linkage

Twelve women in the maternal-health ANC roster do not have a pregnancy-history
record linked to birth-history index 1.

These records are examined separately before previous pregnancy history is
constructed. The purpose is to determine whether they represent valid recent
maternal-health pregnancies that did not result in a live birth.

This distinction is important because the ANC8 outcome is defined from the
maternal-health roster and should not be restricted to live births unless the
EDHS questionnaire structure requires such a restriction.

In [ ]:
# ======================================================================
# AUDIT THE 12 ANC-ROSTER RECORDS WITHOUT AN INDEX LIVE-BIRTH LINK
#
# Purpose:
# Determine why these women have m14_1 recorded but no pidxb_* = 1.
#
# This cell does NOT exclude them and does NOT construct previous
# pregnancy outcome history.
# ======================================================================

no_live_birth_link = (
    linkage_validation_df["n_index_live_birth_links"].eq(0)
)

audit_12 = linkage_validation_df.loc[
    no_live_birth_link
].copy()

print("\n" + "=" * 80)
print("ANC roster records without pidxb_* = 1 linkage")
print("=" * 80)

print("Number of records:", len(audit_12))


# ----------------------------------------------------------------------
# Compare with birth-history availability.
# ----------------------------------------------------------------------

print("\nBirth-order status:")
print(
    audit_12["bord_01"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nANC contact count (m14_1):")

# m14_1 was not carried into the current working dataframe,
# so retrieve it directly from the source IR dataset.
audit_12 = audit_12.merge(
    ir_df[["caseid", "m14_1"]],
    on="caseid",
    how="left",
    validate="one_to_one"
)

print(
    audit_12["m14_1"]
    .value_counts(dropna=False)
    .sort_index()
)


# ----------------------------------------------------------------------
# Show pregnancy outcomes recorded for these 12 women.
#
# p32 coding validated in the previous preprocessing work:
#   1 = born alive
#   2 = born dead
#   3 = miscarriage
#   4 = abortion
# ----------------------------------------------------------------------

p32_labels = {
    1: "Born alive",
    2: "Born dead",
    3: "Miscarriage",
    4: "Abortion"
}

print("\nPregnancy-history outcomes among these 12 women:")

for k, (_, row) in enumerate(audit_12.iterrows()):

    outcomes = []

    for i in range(1, 21):

        value = row[f"p32_{i:02d}"]

        if pd.notna(value):
            outcomes.append(
                f"position {i}: {int(value)} = "
                f"{p32_labels.get(int(value), 'Unknown code')}"
            )

    print(
        f"\nwoman {k + 1} of {len(audit_12)}"
        f"\n  m14_1 ANC contacts: {row['m14_1']}"
        f"\n  bord_01: {row['bord_01']}"
        f"\n  pregnancy outcomes: {', '.join(outcomes)}"
    )

#### 1.6.11.2 Implication for the ANC analytic population

The maternal-health ANC roster is pregnancy-based rather than restricted to
live births.

Twelve women in the roster have no live-birth-history linkage because their
recent recorded pregnancy ended in stillbirth. These women nevertheless have
valid ANC contact information and remain eligible for ANC4+/ANC8+ outcome
ascertainment.

This finding also requires reconsideration of live-birth-specific predictors
such as `bord_01` and `b11_01`. These variables describe the most recent live
birth and may not always correspond to the maternal-health index pregnancy.

Reproductive-history predictors for the ANC8 study will therefore be anchored
to the pregnancy-history record corresponding to the maternal-health index
pregnancy rather than assumed from live-birth-history variables.

#### 1.6.11.3 Identification of the maternal-health index pregnancy

The maternal-health variables with suffix `_1`, including `m14_1`, refer to
the most recent eligible pregnancy covered by the maternal-health roster.

Pregnancy-history position 01 represents the most recent pregnancy outcome,
which may differ from the most recent live birth.

The relationship between pregnancy-history position 01 and the maternal-health
roster is therefore examined directly before pregnancy-order and prior-history
predictors are reconstructed.

In [ ]:
# ======================================================================
# AUDIT PREGNANCY-HISTORY POSITION 01 AS THE MATERNAL-HEALTH INDEX
#
# NOTE:
# p32_01 and pidxb_01 are already present in linkage_validation_df
# from the earlier pregnancy-history merge.
#
# Therefore, only maternal-health variables not already present are
# attached here. This avoids duplicate-column suffixes (_x / _y).
# ======================================================================

index_pregnancy_audit = linkage_validation_df.copy()


# ----------------------------------------------------------------------
# Add only maternal-health variables that are not already present.
# ----------------------------------------------------------------------

maternal_health_vars = [
    "caseid",
    "m14_1",   # number of ANC contacts
    "m10_1",   # pregnancy intention
    "m13_1"    # timing of first ANC
]

# Keep only variables not already available, plus caseid for merging.
vars_to_add = [
    var for var in maternal_health_vars
    if var == "caseid" or var not in index_pregnancy_audit.columns
]

index_pregnancy_audit = index_pregnancy_audit.merge(
    ir_df[vars_to_add],
    on="caseid",
    how="left",
    validate="one_to_one"
)


# ----------------------------------------------------------------------
# Readable p32 pregnancy-outcome labels
# ----------------------------------------------------------------------

p32_labels = {
    1: "Born alive",
    2: "Born dead",
    3: "Miscarriage",
    4: "Abortion"
}

index_pregnancy_audit["p32_01_label"] = (
    index_pregnancy_audit["p32_01"]
    .map(p32_labels)
)


# ----------------------------------------------------------------------
# 1. Pregnancy-history position 01 outcome
# ----------------------------------------------------------------------

print("\n" + "=" * 80)
print("p32_01 = Outcome of pregnancy-history position 01")
print("Position 01 = most recent recorded pregnancy outcome")
print("=" * 80)

print("\nCategory definitions:")
print("  1 = Born alive")
print("  2 = Born dead")
print("  3 = Miscarriage")
print("  4 = Abortion")

print("\nDistribution in the 6,916 maternal-health records:")
print(
    index_pregnancy_audit["p32_01_label"]
    .value_counts(dropna=False)
)


# ----------------------------------------------------------------------
# 2. Relationship between p32_01 and birth-history linkage
# ----------------------------------------------------------------------

print("\n" + "=" * 80)
print("p32_01 outcome by pidxb_01 = birth-history linkage")
print("=" * 80)

print(
    pd.crosstab(
        index_pregnancy_audit["p32_01_label"],
        index_pregnancy_audit["pidxb_01"],
        margins=True,
        dropna=False
    )
)


# ----------------------------------------------------------------------
# 3. ANC-contact information by recent pregnancy outcome
# ----------------------------------------------------------------------

print("\n" + "=" * 80)
print("m14_1 = Number of ANC contacts by p32_01 pregnancy outcome")
print("=" * 80)

print(
    pd.crosstab(
        index_pregnancy_audit["p32_01_label"],
        index_pregnancy_audit["m14_1"],
        margins=True,
        dropna=False
    )
)


# ----------------------------------------------------------------------
# 4. Compact consistency summary
# ----------------------------------------------------------------------

print("\nConsistency summary")
print("-------------------")

print(
    "Maternal-health records:",
    len(index_pregnancy_audit)
)

print(
    "p32_01 = recent pregnancy outcome observed:",
    index_pregnancy_audit["p32_01"].notna().sum()
)

print(
    "m14_1 = ANC contact count observed:",
    index_pregnancy_audit["m14_1"].notna().sum()
)

print(
    "Recent pregnancy not live-born "
    "(p32_01 = born dead / miscarriage / abortion):",
    index_pregnancy_audit["p32_01"].isin([2, 3, 4]).sum()
)

print(
    "Recent pregnancy born alive and also linked to "
    "birth-history index 1:",
    (
        index_pregnancy_audit["p32_01"].eq(1) &
        index_pregnancy_audit["pidxb_01"].eq(1)
    ).sum()
)

#### 1.6.11.4 ANC-specific reproductive-history decision

The audit confirmed that pregnancy-history position 01 corresponds to the
maternal-health index pregnancy for all 6,916 ANC-roster records.

Among these records:

- 6,855 index pregnancies resulted in a live birth;
- 61 resulted in a stillbirth.

Accordingly, the ANC8 study is anchored to the most recent eligible pregnancy
rather than the most recent live birth.

This requires an ANC-specific revision of reproductive-history predictors:

- pregnancy order will be derived from pregnancy history rather than
  `bord_01`;
- preceding pregnancy interval will be evaluated using `p11_01` rather than
  the live-birth-specific `b11_01`;
- previous adverse pregnancy outcome will be derived only from pregnancy-history
  positions 02 onward;
- the outcome of the index pregnancy (`p32_01`) will not be used as a
  predictor because it occurs after ANC contact accumulation.

The earlier `bord_01` and `b11_01` derivations are retained in the notebook as
an audit trail but are superseded for the primary ANC8 model.

In [ ]:
# ======================================================================
# AUDIT ANC-SPECIFIC REPRODUCTIVE-HISTORY VARIABLES
#
# Purpose:
# Replace live-birth-specific reproductive-history constructs with
# pregnancy-based constructs aligned to the ANC index pregnancy.
#
# Candidate variables:
#   Pregnancy order
#       = number of recorded pregnancy-history positions
#
#   p11_01
#       = interval preceding pregnancy-history position 01
#         (the ANC index pregnancy)
#
# No final grouping is performed yet.
# ======================================================================

anc_reproductive_df = index_pregnancy_audit.copy()


# ----------------------------------------------------------------------
# Attach p11_01 if not already present.
# ----------------------------------------------------------------------

if "p11_01" not in anc_reproductive_df.columns:

    anc_reproductive_df = anc_reproductive_df.merge(
        ir_df[["caseid", "p11_01"]],
        on="caseid",
        how="left",
        validate="one_to_one"
    )


# ----------------------------------------------------------------------
# Pregnancy order
#
# n_pregnancy_history_records was already derived from p32_01-p32_20.
# Because position 01 is now validated as the index pregnancy, this count
# represents pregnancy order of the index pregnancy.
# ----------------------------------------------------------------------

anc_reproductive_df["pregnancy_order"] = (
    anc_reproductive_df["n_pregnancy_history_records"]
)


print("\n" + "=" * 78)
print("Derived predictor = Pregnancy order of the ANC index pregnancy")
print("Source = number of recorded p32_01-p32_20 pregnancy-history records")
print("=" * 78)

print("\nDistribution:")
print(
    anc_reproductive_df["pregnancy_order"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "\nMinimum:",
    anc_reproductive_df["pregnancy_order"].min()
)

print(
    "Maximum:",
    anc_reproductive_df["pregnancy_order"].max()
)

print(
    "Missing values:",
    anc_reproductive_df["pregnancy_order"].isna().sum()
)


# ----------------------------------------------------------------------
# p11_01 = preceding pregnancy interval
# ----------------------------------------------------------------------

print("\n" + "=" * 78)
print("p11_01 = Interval preceding the ANC index pregnancy")
print("=" * 78)

print("\nRaw distribution:")
print(
    anc_reproductive_df["p11_01"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nSummary statistics:")
print(
    anc_reproductive_df["p11_01"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

print(
    "\nMissing p11_01 among first pregnancies:",
    (
        anc_reproductive_df["pregnancy_order"].eq(1) &
        anc_reproductive_df["p11_01"].isna()
    ).sum()
)

print(
    "Missing p11_01 among pregnancy order >=2:",
    (
        anc_reproductive_df["pregnancy_order"].ge(2) &
        anc_reproductive_df["p11_01"].isna()
    ).sum()
)

#### 1.6.11.5 ANC-specific pregnancy order and preceding pregnancy interval

Pregnancy-history position 01 has been confirmed as the maternal-health index
pregnancy. Therefore, reproductive-history predictors are reconstructed using
pregnancy history rather than live-birth history.

Pregnancy order is defined as the total number of recorded pregnancy-history
positions for the respondent.

For modelling, pregnancy order is grouped as:

- First pregnancy
- Pregnancy order 2–3
- Pregnancy order 4–5
- Pregnancy order 6+
  
The preceding pregnancy interval is based on `p11_01`, which represents the
interval preceding the index pregnancy.

For `p11_01`:

- first pregnancies are classified explicitly as `First pregnancy`;
- observed intervals are grouped as <24, 24–35, 36–59, and 60+ months;
- the 13 higher-order pregnancies with missing interval information are retained
  as `Unknown / missing`.

These ANC-specific constructs supersede the earlier live-birth-specific
`bord_01` and `b11_01` predictors for the primary ANC8 model.

In [ ]:
# ======================================================================
# CONSTRUCT ANC-SPECIFIC PREGNANCY ORDER AND PRECEDING INTERVAL
#
# Primary ANC8 reproductive-history predictors:
#
#   pregnancy_order_group
#   preceding_pregnancy_interval_group
#
# These replace:
#   bord_01  = live-birth order
#   b11_01   = interval between live births
#
# because the ANC outcome is anchored to the most recent eligible
# pregnancy, not only to pregnancies ending in live birth.
# ======================================================================

anc_reproductive_final = anc_reproductive_df.copy()


# ======================================================================
# 1. GROUP PREGNANCY ORDER
# ======================================================================

anc_reproductive_final["pregnancy_order_group"] = "Unknown / missing"

anc_reproductive_final.loc[
    anc_reproductive_final["pregnancy_order"].eq(1),
    "pregnancy_order_group"
] = "First pregnancy"

anc_reproductive_final.loc[
    anc_reproductive_final["pregnancy_order"].between(2, 3),
    "pregnancy_order_group"
] = "Pregnancy order 2-3"

anc_reproductive_final.loc[
    anc_reproductive_final["pregnancy_order"].between(4, 5),
    "pregnancy_order_group"
] = "Pregnancy order 4-5"

anc_reproductive_final.loc[
    anc_reproductive_final["pregnancy_order"].ge(6),
    "pregnancy_order_group"
] = "Pregnancy order 6+"


pregnancy_order_categories = [
    "First pregnancy",
    "Pregnancy order 2-3",
    "Pregnancy order 4-5",
    "Pregnancy order 6+",
    "Unknown / missing"
]

anc_reproductive_final["pregnancy_order_group"] = pd.Categorical(
    anc_reproductive_final["pregnancy_order_group"],
    categories=pregnancy_order_categories
)


print("\n" + "=" * 78)
print("Derived predictor = Pregnancy order group")
print("=" * 78)

print("\nCategory definitions:")
print("  First pregnancy       = pregnancy order 1")
print("  Pregnancy order 2-3   = pregnancy order 2 or 3")
print("  Pregnancy order 4-5   = pregnancy order 4 or 5")
print("  Pregnancy order 6+    = pregnancy order 6 or higher")
print("  Unknown / missing     = pregnancy order unavailable")

print("\nGrouped distribution:")
print(
    anc_reproductive_final["pregnancy_order_group"]
    .value_counts(dropna=False)
    .reindex(pregnancy_order_categories)
)


# ======================================================================
# 2. GROUP PRECEDING PREGNANCY INTERVAL
# ======================================================================

anc_reproductive_final[
    "preceding_pregnancy_interval_group"
] = "Unknown / missing"


# ----------------------------------------------------------------------
# First pregnancy:
# No preceding pregnancy exists, so p11_01 is structurally not applicable.
# ----------------------------------------------------------------------

anc_reproductive_final.loc[
    anc_reproductive_final["pregnancy_order"].eq(1),
    "preceding_pregnancy_interval_group"
] = "First pregnancy"


# ----------------------------------------------------------------------
# Observed pregnancy intervals
# ----------------------------------------------------------------------

anc_reproductive_final.loc[
    anc_reproductive_final["p11_01"].between(0, 23),
    "preceding_pregnancy_interval_group"
] = "<24 months"

anc_reproductive_final.loc[
    anc_reproductive_final["p11_01"].between(24, 35),
    "preceding_pregnancy_interval_group"
] = "24-35 months"

anc_reproductive_final.loc[
    anc_reproductive_final["p11_01"].between(36, 59),
    "preceding_pregnancy_interval_group"
] = "36-59 months"

anc_reproductive_final.loc[
    anc_reproductive_final["p11_01"].ge(60),
    "preceding_pregnancy_interval_group"
] = "60+ months"


interval_categories = [
    "First pregnancy",
    "<24 months",
    "24-35 months",
    "36-59 months",
    "60+ months",
    "Unknown / missing"
]

anc_reproductive_final[
    "preceding_pregnancy_interval_group"
] = pd.Categorical(
    anc_reproductive_final[
        "preceding_pregnancy_interval_group"
    ],
    categories=interval_categories
)


print("\n" + "=" * 78)
print("p11_01 = Interval preceding the ANC index pregnancy")
print("Derived predictor = Preceding pregnancy interval group")
print("=" * 78)

print("\nCategory definitions:")
print("  First pregnancy       = no preceding pregnancy")
print("  <24 months            = interval 0-23 months")
print("  24-35 months          = interval 24-35 months")
print("  36-59 months          = interval 36-59 months")
print("  60+ months            = interval >=60 months")
print("  Unknown / missing     = higher-order pregnancy with missing p11_01")

print("\nGrouped distribution:")
print(
    anc_reproductive_final[
        "preceding_pregnancy_interval_group"
    ]
    .value_counts(dropna=False)
    .reindex(interval_categories)
)

print(
    "\nResidual unknown/missing among pregnancy order >=2:",
    (
        anc_reproductive_final["pregnancy_order"].ge(2) &
        anc_reproductive_final["p11_01"].isna()
    ).sum()
)

#### 1.6.11.6 Previous adverse pregnancy outcome

Because pregnancy-history position 01 has been validated as the ANC index
pregnancy, previous pregnancy history is derived from positions 02 onward.

The primary derived predictor has three categories:

- No previous pregnancy
- Previous pregnancy, no adverse outcome
- At least one previous adverse outcome

An adverse previous pregnancy outcome is defined as any prior pregnancy coded
as:

- born dead;
- miscarriage; or
- abortion.

Separate supplementary indicators are also retained for each adverse-outcome
type to preserve interpretability and reproducibility.

In [ ]:
# ======================================================================
# CONSTRUCT PREVIOUS ADVERSE PREGNANCY OUTCOME
#
# Source:
#   p32_02 ... p32_20
#
# p32 coding:
#   1 = Born alive
#   2 = Born dead
#   3 = Miscarriage
#   4 = Abortion
#
# Position 01 is the ANC index pregnancy and is therefore excluded from
# previous-pregnancy history.
# ======================================================================

previous_adverse_df = anc_reproductive_final.copy()

prior_p32_vars = [
    f"p32_{i:02d}"
    for i in range(2, 21)
]


# ----------------------------------------------------------------------
# Count the number of previous pregnancies.
#
# Since position 01 is the index pregnancy, any populated p32 position
# from 02 onward represents a pregnancy occurring before the index.
# ----------------------------------------------------------------------

previous_adverse_df["n_previous_pregnancies"] = (
    previous_adverse_df[prior_p32_vars]
    .notna()
    .sum(axis=1)
)


# ----------------------------------------------------------------------
# Supplementary component indicators
# ----------------------------------------------------------------------

previous_adverse_df["previous_born_dead"] = (
    previous_adverse_df[prior_p32_vars]
    .eq(2)
    .any(axis=1)
    .astype(int)
)

previous_adverse_df["previous_miscarriage"] = (
    previous_adverse_df[prior_p32_vars]
    .eq(3)
    .any(axis=1)
    .astype(int)
)

previous_adverse_df["previous_abortion"] = (
    previous_adverse_df[prior_p32_vars]
    .eq(4)
    .any(axis=1)
    .astype(int)
)


# ----------------------------------------------------------------------
# Any previous adverse outcome
# ----------------------------------------------------------------------

previous_adverse_df["any_previous_adverse"] = (
    previous_adverse_df[
        [
            "previous_born_dead",
            "previous_miscarriage",
            "previous_abortion"
        ]
    ]
    .max(axis=1)
)


# ----------------------------------------------------------------------
# Construct primary 3-category predictor
# ----------------------------------------------------------------------

previous_adverse_df["previous_pregnancy_history"] = np.select(
    [
        previous_adverse_df["n_previous_pregnancies"].eq(0),
        (
            previous_adverse_df["n_previous_pregnancies"].gt(0) &
            previous_adverse_df["any_previous_adverse"].eq(0)
        ),
        (
            previous_adverse_df["n_previous_pregnancies"].gt(0) &
            previous_adverse_df["any_previous_adverse"].eq(1)
        )
    ],
    [
        "No previous pregnancy",
        "Previous pregnancy, no adverse outcome",
        "At least one previous adverse outcome"
    ],
    default="Unknown / inconsistent"
)


history_categories = [
    "No previous pregnancy",
    "Previous pregnancy, no adverse outcome",
    "At least one previous adverse outcome",
    "Unknown / inconsistent"
]

previous_adverse_df["previous_pregnancy_history"] = pd.Categorical(
    previous_adverse_df["previous_pregnancy_history"],
    categories=history_categories
)


# ----------------------------------------------------------------------
# Readable audit output
# ----------------------------------------------------------------------

print("\n" + "=" * 80)
print("Derived predictor = Previous pregnancy history")
print("Source = p32_02 to p32_20")
print("=" * 80)

print("\nCategory definitions:")
print("  No previous pregnancy")
print("      = no populated pregnancy-history position after index position 01")
print("")
print("  Previous pregnancy, no adverse outcome")
print("      = at least one previous pregnancy and all prior outcomes born alive")
print("")
print("  At least one previous adverse outcome")
print("      = at least one prior born-dead outcome, miscarriage, or abortion")

print("\nPrimary derived distribution:")
print(
    previous_adverse_df["previous_pregnancy_history"]
    .value_counts(dropna=False)
    .reindex(history_categories)
)


print("\nSupplementary previous adverse-outcome components:")
print(
    pd.DataFrame(
        {
            "Component": [
                "Previous born-dead outcome",
                "Previous miscarriage",
                "Previous abortion",
                "Any previous adverse outcome"
            ],
            "Yes_n": [
                previous_adverse_df["previous_born_dead"].sum(),
                previous_adverse_df["previous_miscarriage"].sum(),
                previous_adverse_df["previous_abortion"].sum(),
                previous_adverse_df["any_previous_adverse"].sum()
            ]
        }
    ).to_string(index=False)
)


print("\nPrevious-pregnancy count summary:")
print(
    previous_adverse_df["n_previous_pregnancies"]
    .describe()
)

print(
    "\nUnknown / inconsistent primary classifications:",
    (
        previous_adverse_df["previous_pregnancy_history"]
        == "Unknown / inconsistent"
    ).sum()
)

The primary previous-pregnancy-history predictor is mutually exclusive, while
the component indicators for prior stillbirth, miscarriage, and abortion are
not mutually exclusive and are retained only as supplementary variables.

### 1.6.12 Maternal employment and occupation

Maternal occupation is measured using `v717`.

The fixed predictor register confirms ten original occupation categories and
complete recording in the EDHS IR dataset.

Because some occupational categories may be sparse in the ANC analytic cohort,
the raw distribution is examined before finalizing the modelling
representation.

In [ ]:
# ======================================================================
# AUDIT MATERNAL OCCUPATION
#
# Variable:
#   v717 = Maternal occupation
#
# Raw category meanings from the fixed EDHS mapping register:
#
#   0 = Not working
#   1 = Professional / technical / managerial
#   2 = Clerical
#   3 = Sales
#   4 = Agricultural self-employed
#   5 = Agricultural employee
#   6 = Household / domestic
#   7 = Services
#   8 = Skilled manual
#   9 = Unskilled manual
#
# Purpose:
# Examine the category sizes in the 6,916-member ANC cohort before
# deciding whether to retain all categories or construct broader groups.
# ======================================================================

maternal_occupation_df = previous_adverse_df.copy()

maternal_occupation_labels = {
    0: "Not working",
    1: "Professional / technical / managerial",
    2: "Clerical",
    3: "Sales",
    4: "Agricultural self-employed",
    5: "Agricultural employee",
    6: "Household / domestic",
    7: "Services",
    8: "Skilled manual",
    9: "Unskilled manual"
}

maternal_occupation_df["maternal_occupation_raw"] = (
    maternal_occupation_df["v717"]
    .map(maternal_occupation_labels)
)


print("\n" + "=" * 80)
print("v717 = Maternal occupation")
print("=" * 80)

print("\nCategory definitions:")
for code, label in maternal_occupation_labels.items():
    print(f"  {code} = {label}")

print("\nRaw coded distribution:")
print(
    maternal_occupation_df["v717"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nLabelled distribution:")
print(
    maternal_occupation_df["maternal_occupation_raw"]
    .value_counts(dropna=False)
)

print(
    "\nUnmapped / missing values:",
    maternal_occupation_df["maternal_occupation_raw"].isna().sum()
)

### 1.6.12 Maternal occupation: preprocessing decision

Maternal occupation (`v717`) contains ten original EDHS occupation categories.

Several categories are sparse in the ANC analytic cohort. The previously
validated preprocessing therefore groups occupations into five broader
categories:

- Not working
- Professional / clerical
- Sales / services
- Agriculture
- Manual / domestic

The original `v717` variable is retained unchanged for reproducibility, while
the grouped variable is used as the primary modelling representation.

In [ ]:
# ======================================================================
# FINALIZE MATERNAL OCCUPATION
#
# Raw variable:
#   v717 = Maternal occupation
#
# Final analytic grouping:
#   Not working
#   Professional / clerical
#   Sales / services
#   Agriculture
#   Manual / domestic
#
# This grouping reuses the previously validated preprocessing decision,
# while all frequencies are re-audited in the current ANC cohort.
# ======================================================================

maternal_occupation_final = maternal_occupation_df.copy()


# ----------------------------------------------------------------------
# Map original EDHS occupation codes into broader analytic groups.
#
# Raw v717 codes:
#   0 = Not working
#   1 = Professional / technical / managerial
#   2 = Clerical
#   3 = Sales
#   4 = Agricultural self-employed
#   5 = Agricultural employee
#   6 = Household / domestic
#   7 = Services
#   8 = Skilled manual
#   9 = Unskilled manual
# ----------------------------------------------------------------------

maternal_occupation_group_map = {
    0: "Not working",

    1: "Professional / clerical",
    2: "Professional / clerical",

    3: "Sales / services",
    7: "Sales / services",

    4: "Agriculture",
    5: "Agriculture",

    6: "Manual / domestic",
    8: "Manual / domestic",
    9: "Manual / domestic"
}

maternal_occupation_final["maternal_occupation_group"] = (
    maternal_occupation_final["v717"]
    .map(maternal_occupation_group_map)
)


# ----------------------------------------------------------------------
# Define explicit category order.
# ----------------------------------------------------------------------

maternal_occupation_categories = [
    "Not working",
    "Professional / clerical",
    "Sales / services",
    "Agriculture",
    "Manual / domestic"
]

maternal_occupation_final["maternal_occupation_group"] = pd.Categorical(
    maternal_occupation_final["maternal_occupation_group"],
    categories=maternal_occupation_categories
)


# ----------------------------------------------------------------------
# Readable demonstration output.
# ----------------------------------------------------------------------

print("\n" + "=" * 80)
print("v717 = Maternal occupation")
print("Derived predictor = maternal_occupation_group")
print("=" * 80)

print("\nFinal category definitions:")
print("  Not working             = raw code 0")
print("  Professional / clerical = raw codes 1, 2")
print("  Sales / services        = raw codes 3, 7")
print("  Agriculture             = raw codes 4, 5")
print("  Manual / domestic       = raw codes 6, 8, 9")

print("\nGrouped distribution:")
print(
    maternal_occupation_final["maternal_occupation_group"]
    .value_counts(dropna=False)
    .reindex(maternal_occupation_categories)
)

print(
    "\nUnmapped / missing values:",
    maternal_occupation_final["maternal_occupation_group"].isna().sum()
)

print(
    "Total:",
    len(maternal_occupation_final)
)

## 1.7 Primary predictor construction audit

Before proceeding further, all primary ANC8 predictor constructs are audited
against the variables actually constructed in the current notebook.

This checkpoint distinguishes:

- constructed and ready predictors;
- constructed predictors with explicit structural categories;
- revised ANC-specific constructs that supersede earlier live-birth-specific
  mappings;
- supplementary representations;
- and constructs still requiring additional work.

This audit does not determine final model inclusion. Final model roles,
redundancy, preprocessing requirements, and leakage considerations will be
reviewed before the predictor dataset is fixed.

In [ ]:
# ======================================================================
# AUDIT ALL PRIMARY ANC8 PREDICTOR CONSTRUCTS
#
# Purpose:
# Reconcile the ANC8-specific eligibility register with the variables that
# have actually been constructed in this notebook.
#
# This is a documentation / quality-control checkpoint.
# It does NOT perform modelling or final predictor selection.
# ======================================================================

predictor_checkpoint_df = maternal_occupation_final.copy()


# ----------------------------------------------------------------------
# Define the ANC8 predictor-construction audit register.
#
# "analytic_variable" refers to the representation currently intended
# for the primary ANC8 analysis.
#
# Important ANC-specific amendments already made:
#
#   v012 replaces maternal age at index birth
#
#   pregnancy_order_group replaces bord_01
#
#   preceding_pregnancy_interval_group replaces b11_01
#
#   previous_pregnancy_history is constructed from p32_02-p32_20
#
# These changes align reproductive-history predictors with the
# pregnancy-based maternal-health ANC roster.
# ----------------------------------------------------------------------

predictor_construction_audit = [
    {
        "construct": "Maternal age",
        "source": "v012",
        "analytic_variable": "v012",
        "representation": "Continuous age at interview",
        "status": "Constructed / ready",
        "note": "ANC-specific amendment: age at interview replaces age at index birth."
    },

    {
        "construct": "Maternal education",
        "source": "v106",
        "analytic_variable": "maternal_education",
        "representation": "4 categories",
        "status": "Constructed / ready",
        "note": "No education, Primary, Secondary, Higher."
    },

    {
        "construct": "Marital / union status",
        "source": "v501",
        "analytic_variable": "marital_status",
        "representation": "6 categories",
        "status": "Constructed / ready",
        "note": "Original EDHS categories retained."
    },

    {
        "construct": "Maternal occupation",
        "source": "v717",
        "analytic_variable": "maternal_occupation_group",
        "representation": "5 grouped categories",
        "status": "Constructed / ready",
        "note": "Not working; Professional/clerical; Sales/services; Agriculture; Manual/domestic."
    },

    {
        "construct": "Partner education",
        "source": "v701",
        "analytic_variable": "partner_education",
        "representation": "Education categories + structural N/A",
        "status": "Constructed / structural N/A preserved",
        "note": "Includes Don't know and Not applicable / not currently in union."
    },

    {
        "construct": "Partner occupation",
        "source": "v705",
        "analytic_variable": "partner_occupation_group",
        "representation": "Grouped categories + structural N/A",
        "status": "Constructed / structural N/A preserved",
        "note": "Includes Unknown and Not applicable / not currently in union."
    },

    {
        "construct": "Healthcare decision-making participation",
        "source": "v743a",
        "analytic_variable": "healthcare_decision_participation",
        "representation": "Participation categories",
        "status": "Constructed / structural N/A preserved",
        "note": "Participates; Does not participate; Not applicable; Unknown/missing."
    },

    {
        "construct": "Household wealth quintile",
        "source": "v190",
        "analytic_variable": "wealth_quintile",
        "representation": "5 quintiles",
        "status": "Constructed / ready",
        "note": "Poorest to Richest."
    },

    {
        "construct": "Place of residence",
        "source": "v025",
        "analytic_variable": "residence",
        "representation": "Urban / Rural",
        "status": "Constructed / ready",
        "note": "Original EDHS coding retained."
    },

    {
        "construct": "Region",
        "source": "v024",
        "analytic_variable": "region",
        "representation": "14 regional categories",
        "status": "Constructed / ready",
        "note": "No collapsing because national geographic representation is important."
    },

    {
        "construct": "Household size",
        "source": "v136",
        "analytic_variable": "household_size",
        "representation": "Continuous count",
        "status": "Constructed / ready",
        "note": "Continuous representation retained."
    },

    {
        "construct": "Religion",
        "source": "v130",
        "analytic_variable": "religion_group",
        "representation": "4 grouped categories",
        "status": "Constructed / ready",
        "note": "Orthodox; Protestant; Muslim; Other religion."
    },

    {
        "construct": "Newspaper / magazine exposure",
        "source": "v157",
        "analytic_variable": "newspaper_exposure",
        "representation": "3 frequency categories",
        "status": "Constructed / ready",
        "note": "Traditional-media domain retained as separate medium."
    },

    {
        "construct": "Radio exposure",
        "source": "v158",
        "analytic_variable": "radio_exposure",
        "representation": "3 frequency categories",
        "status": "Constructed / ready",
        "note": "Traditional-media domain retained as separate medium."
    },

    {
        "construct": "Television exposure",
        "source": "v159",
        "analytic_variable": "television_exposure",
        "representation": "3 frequency categories",
        "status": "Constructed / ready",
        "note": "Traditional-media domain retained as separate medium."
    },

    {
        "construct": "Internet exposure",
        "source": "v171a",
        "analytic_variable": "internet_exposure",
        "representation": "3 timing categories",
        "status": "Constructed / ready",
        "note": "Never; last 12 months; before last 12 months."
    },

    {
        "construct": "Health insurance coverage",
        "source": "v481",
        "analytic_variable": "health_insurance",
        "representation": "Binary",
        "status": "Constructed / ready",
        "note": "No / Yes."
    },

    {
        "construct": "Pregnancy intention",
        "source": "m10_1",
        "analytic_variable": "pregnancy_intention",
        "representation": "3 categories",
        "status": "Constructed / ready",
        "note": "Wanted then; Wanted later; Wanted no more."
    },

    {
        "construct": "Pregnancy order",
        "source": "p32_01-p32_20",
        "analytic_variable": "pregnancy_order_group",
        "representation": "4 grouped categories",
        "status": "Constructed / ANC-specific replacement",
        "note": "Supersedes live-birth-specific bord_01."
    },

    {
        "construct": "Preceding pregnancy interval",
        "source": "p11_01",
        "analytic_variable": "preceding_pregnancy_interval_group",
        "representation": "Interval categories + first pregnancy + unknown",
        "status": "Constructed / ANC-specific replacement",
        "note": "Supersedes live-birth-specific b11_01."
    },

    {
        "construct": "Previous adverse pregnancy outcome",
        "source": "p32_02-p32_20",
        "analytic_variable": "previous_pregnancy_history",
        "representation": "3-category pregnancy-history construct",
        "status": "Constructed / ready",
        "note": "Index pregnancy p32_01 excluded from previous-history ascertainment."
    },

    {
        "construct": "Travel time to nearest hospital / health center",
        "source": "s1101",
        "analytic_variable": "travel_time_min",
        "representation": "Continuous minutes",
        "status": "Constructed / ready",
        "note": "0 retained; 600 retained as top-coded 600+ minutes."
    },

    {
        "construct": "Transport mode to nearest hospital / health center",
        "source": "s1101a",
        "analytic_variable": "transport_mode_group",
        "representation": "4 grouped categories",
        "status": "Constructed / ready",
        "note": "Walking; Motorcycle/bajaj/taxi/minibus; Other motorized; Other non-motorized."
    }
]


audit_df = pd.DataFrame(predictor_construction_audit)


# ----------------------------------------------------------------------
# Verify whether every listed analytic variable is actually present in
# the current working dataframe.
# ----------------------------------------------------------------------

audit_df["variable_present"] = audit_df["analytic_variable"].apply(
    lambda x: x in predictor_checkpoint_df.columns
)


# ----------------------------------------------------------------------
# Calculate missingness for each constructed analytic representation.
# ----------------------------------------------------------------------

def get_missing_n(var):
    if var not in predictor_checkpoint_df.columns:
        return np.nan
    return predictor_checkpoint_df[var].isna().sum()


def get_missing_pct(var):
    if var not in predictor_checkpoint_df.columns:
        return np.nan

    return round(
        predictor_checkpoint_df[var].isna().mean() * 100,
        2
    )


def get_unique_n(var):
    if var not in predictor_checkpoint_df.columns:
        return np.nan

    return predictor_checkpoint_df[var].nunique(dropna=True)


audit_df["missing_n"] = (
    audit_df["analytic_variable"]
    .apply(get_missing_n)
)

audit_df["missing_pct"] = (
    audit_df["analytic_variable"]
    .apply(get_missing_pct)
)

audit_df["unique_non_missing"] = (
    audit_df["analytic_variable"]
    .apply(get_unique_n)
)


# ----------------------------------------------------------------------
# Display the full checkpoint register.
# ----------------------------------------------------------------------

print("\n" + "=" * 110)
print("ANC8 PRIMARY PREDICTOR CONSTRUCTION AUDIT")
print("=" * 110)

print(
    audit_df[
        [
            "construct",
            "source",
            "analytic_variable",
            "representation",
            "status",
            "variable_present",
            "missing_n",
            "missing_pct"
        ]
    ]
    .to_string(index=False)
)


# ----------------------------------------------------------------------
# Compact status summary
# ----------------------------------------------------------------------

print("\n" + "=" * 110)
print("CONSTRUCTION STATUS SUMMARY")
print("=" * 110)

print(
    audit_df["status"]
    .value_counts()
)

print(
    "\nAnalytic variables expected:",
    len(audit_df)
)

print(
    "Analytic variables actually present:",
    audit_df["variable_present"].sum()
)

print(
    "Analytic variables absent:",
    (~audit_df["variable_present"]).sum()
)


# ----------------------------------------------------------------------
# Show only variables with remaining ordinary NaN values.
#
# NOTE:
# Explicit text categories such as:
#   "Not applicable / not currently in union"
#   "Unknown / missing"
# are not counted here as pandas NaN because they were intentionally
# preserved as meaningful analytic categories.
# ----------------------------------------------------------------------

remaining_nan = audit_df.loc[
    audit_df["missing_n"].fillna(0).gt(0),
    [
        "construct",
        "analytic_variable",
        "missing_n",
        "missing_pct",
        "note"
    ]
]

print("\n" + "=" * 110)
print("ANALYTIC VARIABLES WITH REMAINING pandas NaN VALUES")
print("=" * 110)

if len(remaining_nan) == 0:
    print("None.")
else:
    print(
        remaining_nan.to_string(index=False)
    )


# ----------------------------------------------------------------------
# ANC-specific supersession summary
# ----------------------------------------------------------------------

print("\n" + "=" * 110)
print("ANC-SPECIFIC MAPPING AMENDMENTS")
print("=" * 110)

print("1. Maternal age:")
print("   OLD = age at index live birth from v011 + b3_01")
print("   ANC = v012, age at interview")

print("\n2. Reproductive order:")
print("   OLD = bord_01, birth order of most recent live birth")
print("   ANC = pregnancy_order_group from pregnancy history")

print("\n3. Reproductive interval:")
print("   OLD = b11_01, interval between live births")
print("   ANC = preceding_pregnancy_interval_group from p11_01")

print("\n4. Previous adverse pregnancy outcome:")
print("   ANC index pregnancy = p32_01")
print("   Previous history     = p32_02-p32_20 only")

print("\n5. Index pregnancy outcome:")
print("   p32_01 is NOT a predictor because it occurs after ANC accumulation.")

## 1.9 Final reconciliation of the agreed ANC8 predictor set

The following reconciliation is restricted to the 21 predictor constructs
previously agreed for the primary ANC8 model.

It does not discover new predictors and does not reconstruct variables.

Each construct is compared against:

1. the finalized preprocessing register;
2. the variable already constructed in the current ANC8 notebook; and
3. the ANC-specific amendments already justified during this notebook.

Any remaining discrepancy is explicitly flagged before predictor freezing.

In [ ]:
# ======================================================================
# FINAL RECONCILIATION OF THE 21 AGREED ANC8 PREDICTOR CONSTRUCTS
#
# IMPORTANT:
# This cell does NOT create, recode, rename, or modify any variable.
# It is documentation only.
# ======================================================================

anc8_reconciliation = pd.DataFrame([

    {
        "construct": "Maternal age",
        "agreed_source": "v012",
        "current_ANC8_variable": "v012",
        "status": "ANC-specific ready",
        "note": "Use age at interview directly; replaces age-at-index-birth construct."
    },

    {
        "construct": "Maternal education",
        "agreed_source": "v106",
        "current_ANC8_variable": "maternal_education",
        "status": "Ready as built",
        "note": "Four-category education variable already constructed."
    },

    {
        "construct": "Maternal current union status",
        "agreed_source": "v501",
        "current_ANC8_variable": "marital_status",
        "status": "Needs alignment",
        "note": "Current variable has 6 marital-status categories; frozen predictor was binary current union vs not current union."
    },

    {
        "construct": "Maternal occupation",
        "agreed_source": "v717",
        "current_ANC8_variable": "maternal_occupation_group",
        "status": "Ready as built",
        "note": "Validated five-category grouping already constructed."
    },

    {
        "construct": "Partner education",
        "agreed_source": "v701",
        "current_ANC8_variable": "partner_education",
        "status": "Minor label check",
        "note": "Construct already built; confirm Unknown and structural Not applicable labels match frozen convention."
    },

    {
        "construct": "Partner occupation",
        "agreed_source": "v705",
        "current_ANC8_variable": "partner_occupation_group",
        "status": "Minor label check",
        "note": "Construct already built; confirm Unknown and structural Not applicable handling."
    },

    {
        "construct": "Healthcare decision-making participation",
        "agreed_source": "v743a",
        "current_ANC8_variable": "healthcare_decision_participation",
        "status": "Needs alignment",
        "note": "Frozen rule keeps the single applicable unresolved response as true missing, not as a substantive category."
    },

    {
        "construct": "Household wealth quintile",
        "agreed_source": "v190",
        "current_ANC8_variable": "wealth_quintile",
        "status": "Ready as built",
        "note": "Five quintiles already constructed; naming difference only."
    },

    {
        "construct": "Place of residence",
        "agreed_source": "v025",
        "current_ANC8_variable": "residence",
        "status": "Ready as built",
        "note": "Urban/Rural construct already constructed."
    },

    {
        "construct": "Region",
        "agreed_source": "v024",
        "current_ANC8_variable": "region",
        "status": "Ready as built",
        "note": "All 14 regions retained."
    },

    {
        "construct": "Household size",
        "agreed_source": "v136",
        "current_ANC8_variable": "household_size",
        "status": "Ready as built",
        "note": "Continuous household-size count already constructed."
    },

    {
        "construct": "Religion",
        "agreed_source": "v130",
        "current_ANC8_variable": "religion_group",
        "status": "Ready as built",
        "note": "Validated four-category grouping already constructed."
    },

    {
        "construct": "Traditional media exposure",
        "agreed_source": "v157 + v158 + v159",
        "current_ANC8_variable": "NOT YET CONSTRUCTED",
        "status": "Needs construction",
        "note": "Primary predictor must be one traditional-media weekly exposure count (0-3). Newspaper, radio and television are source items only."
    },

    {
        "construct": "Internet exposure",
        "agreed_source": "v171a",
        "current_ANC8_variable": "internet_exposure",
        "status": "Ready as built",
        "note": "Three-category internet history already constructed."
    },

    {
        "construct": "Health insurance coverage",
        "agreed_source": "v481",
        "current_ANC8_variable": "health_insurance",
        "status": "Ready as built",
        "note": "Binary insurance construct already built; naming difference only."
    },

    {
        "construct": "Pregnancy order",
        "agreed_source": "p32_01-p32_20",
        "current_ANC8_variable": "pregnancy_order_group",
        "status": "ANC-specific ready",
        "note": "Pregnancy-based replacement for live-birth order already validated and constructed."
    },

    {
        "construct": "Preceding pregnancy interval",
        "agreed_source": "p11_01",
        "current_ANC8_variable": "preceding_pregnancy_interval_group",
        "status": "ANC-specific ready",
        "note": "Pregnancy-based replacement for preceding live-birth interval already constructed."
    },

    {
        "construct": "Pregnancy intention",
        "agreed_source": "m10_1",
        "current_ANC8_variable": "pregnancy_intention",
        "status": "Ready as built",
        "note": "Three original categories retained."
    },

    {
        "construct": "Previous pregnancy outcome history",
        "agreed_source": "p32_02-p32_20",
        "current_ANC8_variable": "previous_pregnancy_history",
        "status": "ANC-specific ready",
        "note": "Index pregnancy excluded; previous pregnancy history already constructed."
    },

    {
        "construct": "Travel time to nearest hospital / health center",
        "agreed_source": "s1101",
        "current_ANC8_variable": "travel_time_min",
        "status": "Ready as built",
        "note": "Continuous minutes retained; 0 and 600+ handling already validated."
    },

    {
        "construct": "Transport mode to nearest hospital / health center",
        "agreed_source": "s1101a",
        "current_ANC8_variable": "transport_mode_group",
        "status": "Ready as built",
        "note": "Validated four-category grouping already constructed."
    }
])


# ----------------------------------------------------------------------
# Quality-control checks on the reconciliation table itself.
# ----------------------------------------------------------------------

assert len(anc8_reconciliation) == 21
assert anc8_reconciliation["construct"].is_unique


print("\n" + "=" * 115)
print("FINAL RECONCILIATION OF THE 21 AGREED ANC8 PREDICTOR CONSTRUCTS")
print("=" * 115)

print(
    anc8_reconciliation.to_string(index=False)
)


print("\n" + "=" * 115)
print("STATUS SUMMARY")
print("=" * 115)

print(
    anc8_reconciliation["status"]
    .value_counts()
)


print("\nItems requiring action before freeze:")
print(
    anc8_reconciliation.loc[
        anc8_reconciliation["status"].isin(
            ["Needs alignment", "Needs construction", "Minor label check"]
        ),
        [
            "construct",
            "current_ANC8_variable",
            "status",
            "note"
        ]
    ].to_string(index=False)
)

### 1.9.1 Maternal current union status

The six-category `marital_status` variable was already constructed and
validated in the ANC cohort.

For the primary predictor set, the previously fixed binary construct is used:

- **Currently in union** = Married or Living with partner
- **Not currently in union** = Never in union, Widowed, Divorced, or Separated

The original six-category `marital_status` variable is retained in the working
dataframe for auditability but is not used as a separate primary predictor.

In [ ]:
# ======================================================================
# ALIGN MATERNAL CURRENT UNION STATUS WITH THE FROZEN PREDICTOR
#
# IMPORTANT:
# This derives the binary predictor ONLY from the already-cleaned
# marital_status variable.
#
# No raw source variable is reprocessed.
# ======================================================================

anc8_working = maternal_occupation_final.copy()

# ----------------------------------------------------------------------
# Start as missing so any unexpected marital-status label will remain
# visible rather than being silently forced into a category.
# ----------------------------------------------------------------------

anc8_working["maternal_current_union_status"] = np.nan

anc8_working["maternal_current_union_status"] = (
    anc8_working["maternal_current_union_status"]
    .astype("object")
)


# ----------------------------------------------------------------------
# Currently in union
# ----------------------------------------------------------------------

anc8_working.loc[
    anc8_working["marital_status"].isin(
        ["Married", "Living with partner"]
    ),
    "maternal_current_union_status"
] = "Currently in union"


# ----------------------------------------------------------------------
# Not currently in union
# ----------------------------------------------------------------------

anc8_working.loc[
    anc8_working["marital_status"].isin(
        [
            "Never in union",
            "Widowed",
            "Divorced",
            "Separated"
        ]
    ),
    "maternal_current_union_status"
] = "Not currently in union"


# ----------------------------------------------------------------------
# Audit
# ----------------------------------------------------------------------

print("\n" + "=" * 78)
print("Maternal current union status")
print("Derived from already-cleaned marital_status")
print("=" * 78)

print("\nCategory definitions:")
print("  Currently in union     = Married or Living with partner")
print("  Not currently in union = Never in union, Widowed, Divorced, Separated")

print("\nDistribution:")
print(
    anc8_working["maternal_current_union_status"]
    .value_counts(dropna=False)
)

print(
    "\nUnresolved / missing:",
    anc8_working["maternal_current_union_status"].isna().sum()
)


# ----------------------------------------------------------------------
# Cross-check against the six-category source construct.
# ----------------------------------------------------------------------

print("\nCross-tabulation with marital_status:")
print(
    pd.crosstab(
        anc8_working["marital_status"],
        anc8_working["maternal_current_union_status"],
        margins=True,
        dropna=False
    )
)

### 1.9.2 Partner education: structural and label audit

Partner education has already been constructed from `v701`.

This step does not reconstruct the variable. It verifies that:

- substantive education categories occur only where applicable;
- structural non-applicability corresponds to women not currently in union;
- the `Don't know` category represents uncertainty rather than absence of
  partner education.

After verification, only category labels will be standardized to the fixed
predictor terminology if necessary.

In [ ]:
# ======================================================================
# AUDIT PARTNER EDUCATION AGAINST CURRENT UNION STATUS
#
# IMPORTANT:
# No recoding is performed in this cell.
# ======================================================================

print("\n" + "=" * 80)
print("Partner education")
print("Existing variable = partner_education")
print("=" * 80)

print("\nCurrent categories and frequencies:")
print(
    anc8_working["partner_education"]
    .value_counts(dropna=False)
)


# ----------------------------------------------------------------------
# Cross-check category applicability against current union status.
# ----------------------------------------------------------------------

print("\n" + "=" * 80)
print("Partner education by maternal current union status")
print("=" * 80)

print(
    pd.crosstab(
        anc8_working["partner_education"],
        anc8_working["maternal_current_union_status"],
        margins=True,
        dropna=False
    )
)


# ----------------------------------------------------------------------
# Targeted consistency counts
# ----------------------------------------------------------------------

n_not_applicable_in_union = (
    anc8_working["maternal_current_union_status"]
    .eq("Currently in union")
    &
    anc8_working["partner_education"]
    .eq("Not applicable / not currently in union")
).sum()

n_substantive_outside_union = (
    anc8_working["maternal_current_union_status"]
    .eq("Not currently in union")
    &
    ~anc8_working["partner_education"]
    .eq("Not applicable / not currently in union")
).sum()

n_dont_know = (
    anc8_working["partner_education"]
    .eq("Don't know")
).sum()


print("\nConsistency checks")
print("------------------")

print(
    "Not-applicable partner education among currently-in-union women:",
    n_not_applicable_in_union
)

print(
    "Substantive/unknown partner education among women not currently in union:",
    n_substantive_outside_union
)

print(
    "Don't know partner education responses:",
    n_dont_know
)

#### Partner education: final label standardization

The existing partner-education construct is internally consistent with current
union status.

Only category labels are standardized to match the fixed preprocessing
register:

- `Don't know` → `Unknown`
- `Not applicable / not currently in union` → `Not applicable`

No substantive category is changed.

In [ ]:
# ======================================================================
# STANDARDIZE PARTNER-EDUCATION LABELS
#
# IMPORTANT:
# This does not reconstruct partner education.
# It only standardizes two existing labels to the fixed convention.
# ======================================================================

anc8_working["partner_education"] = (
    anc8_working["partner_education"]
    .replace({
        "Don't know": "Unknown",
        "Not applicable / not currently in union": "Not applicable"
    })
)

print("\n" + "=" * 78)
print("Partner education: final standardized labels")
print("=" * 78)

print(
    anc8_working["partner_education"]
    .value_counts(dropna=False)
)

print(
    "\nMissing values:",
    anc8_working["partner_education"].isna().sum()
)

### 1.9.3 Partner occupation: structural and label audit

Partner occupation has already been constructed from `v705`.

This step verifies that:

- substantive occupation categories occur only among women currently in union;
- structural non-applicability corresponds exactly to women not currently in union;
- `Unknown` remains distinct from structural non-applicability.

No substantive recoding is performed unless an inconsistency is identified.

In [ ]:
# ======================================================================
# AUDIT PARTNER OCCUPATION AGAINST CURRENT UNION STATUS
#
# IMPORTANT:
# No reconstruction or regrouping occurs in this cell.
# ======================================================================

print("\n" + "=" * 80)
print("Partner occupation")
print("Existing variable = partner_occupation_group")
print("=" * 80)

print("\nCurrent categories and frequencies:")
print(
    anc8_working["partner_occupation_group"]
    .value_counts(dropna=False)
)


# ----------------------------------------------------------------------
# Cross-check structural applicability against current union status.
# ----------------------------------------------------------------------

print("\n" + "=" * 80)
print("Partner occupation by maternal current union status")
print("=" * 80)

print(
    pd.crosstab(
        anc8_working["partner_occupation_group"],
        anc8_working["maternal_current_union_status"],
        margins=True,
        dropna=False
    )
)


# ----------------------------------------------------------------------
# Targeted consistency checks
# ----------------------------------------------------------------------

n_not_applicable_in_union = (
    anc8_working["maternal_current_union_status"]
    .eq("Currently in union")
    &
    anc8_working["partner_occupation_group"]
    .eq("Not applicable / not currently in union")
).sum()

n_substantive_outside_union = (
    anc8_working["maternal_current_union_status"]
    .eq("Not currently in union")
    &
    ~anc8_working["partner_occupation_group"]
    .eq("Not applicable / not currently in union")
).sum()

n_unknown = (
    anc8_working["partner_occupation_group"]
    .eq("Unknown")
).sum()


print("\nConsistency checks")
print("------------------")

print(
    "Not-applicable partner occupation among currently-in-union women:",
    n_not_applicable_in_union
)

print(
    "Substantive/unknown partner occupation among women not currently in union:",
    n_substantive_outside_union
)

print(
    "Unknown partner occupation responses:",
    n_unknown
)

#### Partner occupation: final label standardization

The existing partner-occupation construct is internally consistent with current
union status.

Only the structural category label is standardized to match the fixed
preprocessing register:

- `Not applicable / not currently in union` → `Not applicable`

No substantive occupation grouping is changed.

In [ ]:
# ======================================================================
# STANDARDIZE PARTNER-OCCUPATION LABEL
#
# IMPORTANT:
# This does not reconstruct or regroup partner occupation.
# It changes only the structural category label.
# ======================================================================

anc8_working["partner_occupation_group"] = (
    anc8_working["partner_occupation_group"]
    .astype(object)  # pandas 3: relabel as text
    .replace({
        "Not applicable / not currently in union": "Not applicable"
    })
)

print("\n" + "=" * 78)
print("Partner occupation: final standardized labels")
print("=" * 78)

print(
    anc8_working["partner_occupation_group"]
    .value_counts(dropna=False)
)

print(
    "\nMissing values:",
    anc8_working["partner_occupation_group"].isna().sum()
)

### 1.9.4 Healthcare decision-making participation: final audit

Healthcare decision-making participation has already been constructed.

The fixed preprocessing rule distinguishes:

- **Participates**
- **Does not participate**
- **Not applicable** for women not currently in union
- true missing for the single applicable woman with unresolved decision information

This step first verifies the existing construction against current union status
before the final label/missing-value alignment is applied.

In [ ]:
# ======================================================================
# AUDIT HEALTHCARE DECISION-MAKING PARTICIPATION
#
# IMPORTANT:
# No reconstruction is performed here.
# We inspect only the already-constructed variable.
# ======================================================================

print("\n" + "=" * 80)
print("Healthcare decision-making participation")
print("Existing variable = healthcare_decision_participation")
print("=" * 80)

print("\nCurrent categories and frequencies:")
print(
    anc8_working["healthcare_decision_participation"]
    .value_counts(dropna=False)
)


# ----------------------------------------------------------------------
# Cross-check against current union status.
# ----------------------------------------------------------------------

print("\n" + "=" * 80)
print("Healthcare decision participation by maternal current union status")
print("=" * 80)

print(
    pd.crosstab(
        anc8_working["healthcare_decision_participation"],
        anc8_working["maternal_current_union_status"],
        margins=True,
        dropna=False
    )
)


# ----------------------------------------------------------------------
# Targeted consistency checks
# ----------------------------------------------------------------------

n_not_applicable_in_union = (
    anc8_working["maternal_current_union_status"].eq("Currently in union")
    &
    anc8_working["healthcare_decision_participation"]
        .eq("Not applicable / not currently in union")
).sum()

n_substantive_outside_union = (
    anc8_working["maternal_current_union_status"].eq("Not currently in union")
    &
    ~anc8_working["healthcare_decision_participation"]
        .eq("Not applicable / not currently in union")
).sum()

n_unknown_category = (
    anc8_working["healthcare_decision_participation"]
    .eq("Unknown / missing")
).sum()

n_true_nan = (
    anc8_working["healthcare_decision_participation"]
    .isna()
    .sum()
)


print("\nConsistency checks")
print("------------------")

print(
    "Not-applicable category among currently-in-union women:",
    n_not_applicable_in_union
)

print(
    "Substantive/unknown category among women not currently in union:",
    n_substantive_outside_union
)

print(
    "Current 'Unknown / missing' category count:",
    n_unknown_category
)

print(
    "Current true NaN count:",
    n_true_nan
)

#### Healthcare decision-making participation: final alignment

The existing healthcare decision-making construct was internally consistent.

Final alignment preserves:

- `Participates`
- `Does not participate`
- `Not applicable` for women not currently in union

The single applicable unresolved response is represented as true missing
rather than as a substantive category.

No participation classification is otherwise changed.

In [ ]:
# ======================================================================
# FINAL ALIGNMENT OF HEALTHCARE DECISION-MAKING PARTICIPATION
#
# The existing variable is categorical. Convert only its dtype to object
# so that the unresolved category can safely become a true NaN.
#
# No substantive classification is reconstructed.
# ======================================================================

# Preserve the already-constructed values, but remove categorical dtype
# restrictions before changing labels / missing representation.
anc8_working["healthcare_decision_participation"] = (
    anc8_working["healthcare_decision_participation"]
    .astype("object")
)

# Standardize the structural label.
anc8_working.loc[
    anc8_working["healthcare_decision_participation"]
    .eq("Not applicable / not currently in union"),
    "healthcare_decision_participation"
] = "Not applicable"

# Convert the single applicable unresolved response to true missing.
anc8_working.loc[
    anc8_working["healthcare_decision_participation"]
    .eq("Unknown / missing"),
    "healthcare_decision_participation"
] = np.nan


# ----------------------------------------------------------------------
# Final audit
# ----------------------------------------------------------------------

print("\n" + "=" * 82)
print("Healthcare decision-making participation: final aligned representation")
print("=" * 82)

print(
    anc8_working["healthcare_decision_participation"]
    .value_counts(dropna=False)
)

print(
    "\nTrue missing values:",
    anc8_working["healthcare_decision_participation"].isna().sum()
)

print(
    "Final dtype:",
    anc8_working["healthcare_decision_participation"].dtype
)

### 1.9.5 Traditional media exposure: source-variable audit

The three traditional-media source variables were already cleaned earlier in
this notebook:

- `newspaper_exposure`
- `radio_exposure`
- `television_exposure`

They are not separate primary predictors.

Before constructing the agreed traditional-media composite, their existing
category labels and distributions are verified. The composite will then be
derived only from these already-cleaned variables.

In [ ]:
# ======================================================================
# AUDIT THE ALREADY-CLEANED TRADITIONAL-MEDIA SOURCE VARIABLES
#
# IMPORTANT:
# - No new predictor is created here.
# - No raw EDHS variable is reprocessed.
# - We only inspect the three previously cleaned source variables.
# ======================================================================

media_source_vars = [
    "newspaper_exposure",
    "radio_exposure",
    "television_exposure"
]

print("\n" + "=" * 90)
print("TRADITIONAL-MEDIA SOURCE VARIABLES: EXISTING CLEANED REPRESENTATIONS")
print("=" * 90)

for var in media_source_vars:

    print(f"\n{var}")
    print("-" * len(var))

    print(
        anc8_working[var]
        .value_counts(dropna=False)
    )

    print(
        "Missing values:",
        anc8_working[var].isna().sum()
    )

    print(
        "Dtype:",
        anc8_working[var].dtype
    )

**Two last steps, and the analysis checkpoint.** The study's copy of this notebook does not
hold the cells for two steps it relies on below; they are restored here, and the fingerprint
check at the end of the notebook confirms that they reproduce the study's frame exactly:

- the occupation labels are written without spaces around the slash ("Sales/services");
- **`traditional_media_weekly_count`**, one of the 21, is the number of the three traditional
  media (newspaper, radio, television) the woman uses **at least once a week**, from 0 to 3.
  The audit just above shows the three cleaned exposure variables it counts.

The aligned working frame is then kept as `anc8_final`, the checkpoint the audit and the
export below start from.

In [ ]:
for var in ["maternal_occupation_group", "partner_occupation_group"]:
    anc8_working[var] = anc8_working[var].astype(object).str.replace(" / ", "/", regex=False)

media_source_vars = ["newspaper_exposure", "radio_exposure", "television_exposure"]
anc8_working["traditional_media_weekly_count"] = sum(
    anc8_working[v].eq("At least once a week").astype(int) for v in media_source_vars
)
print(anc8_working["traditional_media_weekly_count"].value_counts().sort_index().to_string())

anc8_final = anc8_working.copy()

In [ ]:
# ================================================================
# CHECK WHETHER THE LATER ANC8 CHECKPOINT ALREADY CONTAINS
# THE VARIABLES WE HAVE JUST ALIGNED.
#
# No modification is performed.
# ================================================================

check_vars = [
    "maternal_current_union_status",
    "partner_education",
    "partner_occupation_group",
    "healthcare_decision_participation",
    "traditional_media_weekly_count"
]

for var in check_vars:
    print(
        f"{var}:",
        "PRESENT" if var in anc8_final.columns else "NOT PRESENT"
    )

In [ ]:
# ======================================================================
# FINAL INTEGRITY AUDIT OF THE 21 AGREED ANC8 PRIMARY PREDICTORS
#
# Source dataframe:
#   anc8_final
#
# IMPORTANT:
# - No variable is created.
# - No variable is recoded.
# - No variable is renamed.
# - This is inspection only.
# ======================================================================

anc8_working = anc8_final.copy()

anc8_primary_predictors = [
    "v012",
    "maternal_education",
    "maternal_current_union_status",
    "maternal_occupation_group",
    "partner_education",
    "partner_occupation_group",
    "healthcare_decision_participation",
    "wealth_quintile",
    "residence",
    "region",
    "household_size",
    "religion_group",
    "traditional_media_weekly_count",
    "internet_exposure",
    "health_insurance",
    "pregnancy_order_group",
    "preceding_pregnancy_interval_group",
    "pregnancy_intention",
    "previous_pregnancy_history",
    "travel_time_min",
    "transport_mode_group"
]

# ------------------------------------------------------------
# Basic structural checks
# ------------------------------------------------------------

assert len(anc8_primary_predictors) == 21
assert len(set(anc8_primary_predictors)) == 21

missing_columns = [
    var for var in anc8_primary_predictors
    if var not in anc8_working.columns
]

print("=" * 100)
print("FINAL ANC8 PRIMARY-PREDICTOR INTEGRITY AUDIT")
print("=" * 100)

print("\nRows in working dataframe:", len(anc8_working))
print("Expected primary predictors:", len(anc8_primary_predictors))
print("Missing predictor columns:", missing_columns)


# ------------------------------------------------------------
# Variable-level audit
# ------------------------------------------------------------

audit_rows = []

for var in anc8_primary_predictors:

    s = anc8_working[var]

    audit_rows.append({
        "variable": var,
        "dtype": str(s.dtype),
        "non_missing_n": int(s.notna().sum()),
        "missing_n": int(s.isna().sum()),
        "unique_non_missing": int(s.nunique(dropna=True))
    })

final_predictor_audit = pd.DataFrame(audit_rows)

print("\nVariable-level summary:")
print(
    final_predictor_audit.to_string(index=False)
)


# ------------------------------------------------------------
# Category / value audit
#
# Printed separately so every predictor can be interpreted without
# needing to return to an earlier notebook cell.
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PREDICTOR VALUES / CATEGORY MEANINGS")
print("=" * 100)

for var in anc8_primary_predictors:

    print(f"\n{var}")
    print("-" * len(var))

    s = anc8_working[var]

    # Continuous/count variables: show numeric summary.
    if var in [
        "v012",
        "household_size",
        "traditional_media_weekly_count",
        "travel_time_min"
    ]:

        print(
            s.describe(
                percentiles=[0.25, 0.50, 0.75]
            )
        )

    # Categorical variables: show complete category frequencies.
    else:

        print(
            s.value_counts(
                dropna=False
            )
        )

In [ ]:
# ================================================================
# SURVEY-WEIGHTED RELIGION DISTRIBUTION
# ANC8 analytic cohort only: n = 6,916
#
# v005 is brought from the original IR dataset using caseid.
# No predictor is changed.
# ================================================================

religion_labels = {
    1: "Orthodox",
    2: "Catholic",
    3: "Protestant",
    4: "Muslim",
    5: "Traditional",
    96: "Other"
}

# Keep only identifier and sampling weight from the original IR dataset.
weights_df = ir_df[["caseid", "v005"]].copy()

# Merge survey weight onto the already-defined ANC8 cohort.
religion_weighted = (
    anc8_working[["caseid", "v130"]]
    .merge(
        weights_df,
        on="caseid",
        how="left",
        validate="one_to_one"
    )
)

# DHS sampling weight.
religion_weighted["weight"] = (
    religion_weighted["v005"] / 1_000_000
)

religion_weighted["religion"] = (
    religion_weighted["v130"]
    .map(religion_labels)
)

# ---------------------------------------------------------------
# Quality checks
# ---------------------------------------------------------------

print("Rows:", len(religion_weighted))
print("Missing v005:", religion_weighted["v005"].isna().sum())

# ---------------------------------------------------------------
# Weighted religion distribution
# ---------------------------------------------------------------

weighted_sum = (
    religion_weighted
    .groupby("religion", observed=False)["weight"]
    .sum()
)

weighted_percent = (
    weighted_sum / weighted_sum.sum() * 100
)

weighted_result = pd.DataFrame({
    "weighted_sum": weighted_sum.round(2),
    "weighted_percent": weighted_percent.round(2)
}).sort_values(
    "weighted_percent",
    ascending=False
)

print("\nSurvey-weighted religion distribution:")
print(weighted_result)

In [ ]:
[
    (name, obj.shape)
    for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
    and {"v130", "v005"}.issubset(obj.columns)
]

In [ ]:
# =============================================================================
# ANC CONTACT COUNT: DESCRIPTIVE SUMMARY AND WEIGHTED DISTRIBUTION FIGURE
#
# ANC cohort:
#   anc8_final
#
# Survey weight:
#   v005 retrieved from the original IR dataset (ir_df) using caseid
#
# Outcome source:
#   m14_1 = reported number of ANC contacts
#
# Rules:
#   - m14_1 = 98 ("don't know") is unresolved and excluded
#   - 0 ANC contacts is a valid substantive value and is retained
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ---------------------------------------------------------------------
# 1. Output folder
# ---------------------------------------------------------------------

output_dir = PROJECT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)



# ---------------------------------------------------------------------
# 2. Start from the already-defined ANC8 cohort.
#
# v005 is NOT stored in anc8_final, so retrieve it from ir_df by caseid.
# ---------------------------------------------------------------------

anc_count_df = (
    anc8_final[["caseid", "m14_1"]]
    .merge(
        ir_df[["caseid", "v005"]],
        on="caseid",
        how="left",
        validate="one_to_one"
    )
)


# ---------------------------------------------------------------------
# 3. Integrity checks before analysis
# ---------------------------------------------------------------------

assert len(anc_count_df) == 6916, (
    f"Expected 6916 ANC roster records, found {len(anc_count_df)}."
)

assert anc_count_df["caseid"].is_unique, (
    "caseid is not unique after merging survey weights."
)

assert anc_count_df["v005"].notna().all(), (
    "Some ANC records did not receive a survey weight."
)

print("ANC roster records:", len(anc_count_df))
print("Missing survey weights:", anc_count_df["v005"].isna().sum())


# ---------------------------------------------------------------------
# 4. Resolve ANC contact count.
#
# m14_1 = 98 is DHS 'don't know' and is not interpreted as 98 visits.
# ---------------------------------------------------------------------

anc_count_df["anc_contacts"] = pd.to_numeric(
    anc_count_df["m14_1"],
    errors="coerce"
)

anc_count_df.loc[
    anc_count_df["anc_contacts"].eq(98),
    "anc_contacts"
] = np.nan

anc_count_df["weight"] = anc_count_df["v005"] / 1_000_000


# Resolved count observations only
anc_count_resolved = (
    anc_count_df[
        anc_count_df["anc_contacts"].notna()
    ]
    .copy()
)

anc_count_resolved["anc_contacts"] = (
    anc_count_resolved["anc_contacts"]
    .astype(int)
)


# ---------------------------------------------------------------------
# 5. Outcome accounting
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("ANC CONTACT COUNT ACCOUNTING")
print("=" * 75)

print("ANC roster denominator:", len(anc_count_df))
print("Resolved ANC count:", len(anc_count_resolved))
print(
    "Unresolved ANC count:",
    anc_count_df["anc_contacts"].isna().sum()
)
print(
    "Zero ANC contacts:",
    anc_count_resolved["anc_contacts"].eq(0).sum()
)


# ---------------------------------------------------------------------
# 6. Descriptive statistics
# ---------------------------------------------------------------------

mean_unweighted = anc_count_resolved["anc_contacts"].mean()
sd_unweighted = anc_count_resolved["anc_contacts"].std()

median_unweighted = anc_count_resolved["anc_contacts"].median()
q1 = anc_count_resolved["anc_contacts"].quantile(0.25)
q3 = anc_count_resolved["anc_contacts"].quantile(0.75)

minimum = anc_count_resolved["anc_contacts"].min()
maximum = anc_count_resolved["anc_contacts"].max()

mean_weighted = np.average(
    anc_count_resolved["anc_contacts"],
    weights=anc_count_resolved["weight"]
)


print("\n" + "=" * 75)
print("ANC CONTACT COUNT DESCRIPTIVE SUMMARY")
print("=" * 75)

print(f"Resolved denominator: {len(anc_count_resolved)}")
print(f"Unweighted mean (SD): {mean_unweighted:.2f} ({sd_unweighted:.2f})")
print(
    f"Unweighted median (IQR): "
    f"{median_unweighted:.0f} ({q1:.0f}-{q3:.0f})"
)
print(f"Observed range: {minimum}-{maximum}")
print(f"Survey-weighted mean: {mean_weighted:.2f}")


# ---------------------------------------------------------------------
# 7. Save descriptive summary
# ---------------------------------------------------------------------

anc_contact_summary = pd.DataFrame([{
    "ANC_roster_n": len(anc_count_df),
    "resolved_n": len(anc_count_resolved),
    "unresolved_n": anc_count_df["anc_contacts"].isna().sum(),
    "zero_ANC_n": anc_count_resolved["anc_contacts"].eq(0).sum(),
    "unweighted_mean": round(mean_unweighted, 3),
    "unweighted_sd": round(sd_unweighted, 3),
    "unweighted_median": median_unweighted,
    "unweighted_Q1": q1,
    "unweighted_Q3": q3,
    "minimum": minimum,
    "maximum": maximum,
    "survey_weighted_mean": round(mean_weighted, 3)
}])

summary_path = output_dir / "ANC_contact_count_descriptive_summary.csv"
anc_contact_summary.to_csv(summary_path, index=False)


# ---------------------------------------------------------------------
# 8. Survey-weighted distribution by exact ANC contact count
# ---------------------------------------------------------------------

weighted_distribution = (
    anc_count_resolved
    .groupby("anc_contacts", as_index=False)["weight"]
    .sum()
    .rename(columns={"weight": "weighted_sum"})
)

weighted_distribution["weighted_percent"] = (
    weighted_distribution["weighted_sum"]
    / weighted_distribution["weighted_sum"].sum()
    * 100
)

distribution_path = (
    output_dir / "ANC_contact_count_weighted_distribution.csv"
)

weighted_distribution.to_csv(
    distribution_path,
    index=False
)


# ---------------------------------------------------------------------
# 9. High-level figure
#
# A bar chart is used because ANC contact count is a discrete integer
# variable rather than a continuous measurement.
# ---------------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.bar(
    weighted_distribution["anc_contacts"],
    weighted_distribution["weighted_percent"]
)

plt.xlabel("Number of ANC contacts")
plt.ylabel("Survey-weighted percentage of women")
plt.title("Distribution of antenatal care contacts in Ethiopia")

plt.xticks(
    weighted_distribution["anc_contacts"]
)

plt.tight_layout()

figure_path = (
    output_dir / "Figure_ANC_contact_count_weighted_distribution.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------------------
# 10. Export confirmation
# ---------------------------------------------------------------------

print("\nExported:")
print(summary_path)
print(distribution_path)
print(figure_path)

In [ ]:
# =============================================================================
# FREEZE ANC8 ANALYTIC HANDOFF DATASET FOR the later notebooks
#
# Purpose:
# Create the single authoritative modelling dataset that the later notebooks will use.
#
# Contains only:
#   - caseid
#   - ANC4+ outcome
#   - ANC8+ outcome
#   - 21 agreed primary predictors
#   - survey weight, PSU, and stratum
#
# No predictor reconstruction should occur in the later notebooks.
# =============================================================================

from pathlib import Path
import hashlib
import pandas as pd

output_dir = PROJECT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# 1. Authoritative primary predictor list
# ---------------------------------------------------------------------

anc8_primary_predictors = [
    "v012",
    "maternal_education",
    "maternal_current_union_status",
    "maternal_occupation_group",
    "partner_education",
    "partner_occupation_group",
    "healthcare_decision_participation",
    "wealth_quintile",
    "residence",
    "region",
    "household_size",
    "religion_group",
    "traditional_media_weekly_count",
    "internet_exposure",
    "health_insurance",
    "pregnancy_order_group",
    "preceding_pregnancy_interval_group",
    "pregnancy_intention",
    "previous_pregnancy_history",
    "travel_time_min",
    "transport_mode_group"
]


# ---------------------------------------------------------------------
# 2. Bring survey design variables from original IR dataset
# ---------------------------------------------------------------------

survey_design = ir_df[
    ["caseid", "v005", "v021", "v022"]
].copy()


# ---------------------------------------------------------------------
# 3. Build fixed analytic handoff dataset
# ---------------------------------------------------------------------

anc8_handoff = (
    anc8_final[
        ["caseid", "anc4plus", "anc8plus"] + anc8_primary_predictors
    ]
    .merge(
        survey_design,
        on="caseid",
        how="left",
        validate="one_to_one"
    )
)


# ---------------------------------------------------------------------
# 4. Freeze-readiness assertions
# ---------------------------------------------------------------------

assert len(anc8_handoff) == 6916
assert anc8_handoff["caseid"].is_unique

assert len(anc8_primary_predictors) == 21
assert len(set(anc8_primary_predictors)) == 21

assert anc8_handoff["v005"].notna().all()
assert anc8_handoff["v021"].notna().all()
assert anc8_handoff["v022"].notna().all()

assert anc8_handoff["anc8plus"].isna().sum() == 23
assert anc8_handoff["anc4plus"].isna().sum() == 23

assert anc8_handoff[
    "traditional_media_weekly_count"
].between(0, 3).all()

assert anc8_handoff[
    "healthcare_decision_participation"
].isna().sum() == 1


# ---------------------------------------------------------------------
# 5. Export PKL
# ---------------------------------------------------------------------

pkl_path = output_dir / "ANC8_analysis_frozen.pkl"

anc8_handoff.to_pickle(pkl_path)


# ---------------------------------------------------------------------
# 6. Compute SHA-256 hash
# ---------------------------------------------------------------------

sha256 = hashlib.sha256()

with open(pkl_path, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)

pkl_hash = sha256.hexdigest()


# ---------------------------------------------------------------------
# 7. Confirmation
# ---------------------------------------------------------------------

print("\n" + "=" * 85)
print("ANC8 FROZEN ANALYTIC HANDOFF DATASET")
print("=" * 85)

print("Rows:", anc8_handoff.shape[0])
print("Columns:", anc8_handoff.shape[1])
print("Primary predictors:", len(anc8_primary_predictors))

print("\nANC8 unresolved:", anc8_handoff["anc8plus"].isna().sum())
print("ANC4 unresolved:", anc8_handoff["anc4plus"].isna().sum())

print("\nSurvey variables:")
print("  v005 = sampling weight")
print("  v021 = PSU")
print("  v022 = stratum")

print("\nSaved to:")
print(pkl_path)

print("\nSHA-256:")
print(pkl_hash)

In [ ]:
# =============================================================================
# EXPORT ANC8 NOTEBOOK-1 ARTIFACT MANIFEST
#
# Purpose:
# Record the key fixed outputs from this notebook together with their purpose,
# dimensions where relevant, and SHA-256 hashes.
#
# This manifest becomes the reproducibility governor for the later notebooks.
# =============================================================================

from pathlib import Path
import hashlib
import pandas as pd

output_dir = PROJECT / "outputs"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


# ---------------------------------------------------------------------
# Files already created in this notebook
# ---------------------------------------------------------------------

artifact_paths = {
    "ANC8 frozen analytic dataset":
        output_dir / "ANC8_analysis_frozen.pkl",

    "ANC contact descriptive summary":
        output_dir / "ANC_contact_count_descriptive_summary.csv",

    "ANC contact weighted distribution":
        output_dir / "ANC_contact_count_weighted_distribution.csv",

    "ANC contact distribution figure":
        output_dir / "Figure_ANC_contact_count_weighted_distribution.png",
}


# ---------------------------------------------------------------------
# Build manifest
# ---------------------------------------------------------------------

manifest_rows = []

for artifact_name, path in artifact_paths.items():

    exists = path.exists()

    manifest_rows.append({
        "artifact": artifact_name,
        "filename": path.name,
        "exists": exists,
        "sha256": sha256_file(path) if exists else None
    })

artifact_manifest = pd.DataFrame(manifest_rows)


# Add fixed dataset dimensions explicitly
artifact_manifest.loc[
    artifact_manifest["artifact"].eq("ANC8 frozen analytic dataset"),
    "rows"
] = anc8_handoff.shape[0]

artifact_manifest.loc[
    artifact_manifest["artifact"].eq("ANC8 frozen analytic dataset"),
    "columns"
] = anc8_handoff.shape[1]


# ---------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------

manifest_path = output_dir / "ANC8_artifact_manifest.csv"

artifact_manifest.to_csv(
    manifest_path,
    index=False
)


# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print("\n" + "=" * 95)
print("ANC8 NOTEBOOK-1 ARTIFACT MANIFEST")
print("=" * 95)

print(
    artifact_manifest.to_string(index=False)
)

print("\nManifest saved to:")
print(manifest_path)

In [ ]:
# =============================================================================
# EXPORT FINAL ANC8 PREDICTOR REGISTER
#
# Purpose:
# Document the exact 21 predictors fixed in ANC8_analysis_frozen.pkl.
#
# IMPORTANT:
# - No predictor is created or modified here.
# - The register describes the already-finalized ANC8 predictors.
# =============================================================================

from pathlib import Path
import pandas as pd

output_dir = PROJECT / "outputs"


predictor_register = pd.DataFrame([

    {
        "predictor_id": 1,
        "domain": "Maternal and sociodemographic",
        "construct": "Maternal age",
        "source_variables": "v012",
        "final_variable": "v012",
        "final_type": "Continuous",
        "final_coding": "Age at interview in completed years",
        "ANC8_decision": "ANC-specific amendment",
        "missing_handling": "No missing values",
        "notes": "Replaces age at index live birth because the ANC8 cohort is pregnancy-based."
    },

    {
        "predictor_id": 2,
        "domain": "Maternal and sociodemographic",
        "construct": "Maternal educational attainment",
        "source_variables": "v106",
        "final_variable": "maternal_education",
        "final_type": "Ordinal categorical",
        "final_coding": "No education; Primary; Secondary; Higher",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": "Previously validated four-category construct."
    },

    {
        "predictor_id": 3,
        "domain": "Maternal and sociodemographic",
        "construct": "Maternal current union status",
        "source_variables": "v501",
        "final_variable": "maternal_current_union_status",
        "final_type": "Binary categorical",
        "final_coding": "Currently in union; Not currently in union",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": "Currently in union = married or living with partner."
    },

    {
        "predictor_id": 4,
        "domain": "Maternal and sociodemographic",
        "construct": "Maternal occupation",
        "source_variables": "v717",
        "final_variable": "maternal_occupation_group",
        "final_type": "Nominal categorical",
        "final_coding": "Not working; Professional/clerical; Sales/services; Agriculture; Manual/domestic",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": "Previously validated grouped occupation construct."
    },

    {
        "predictor_id": 5,
        "domain": "Maternal and sociodemographic",
        "construct": "Partner education",
        "source_variables": "v701",
        "final_variable": "partner_education",
        "final_type": "Nominal categorical",
        "final_coding": "No education; Primary; Secondary; Higher; Unknown; Not applicable",
        "ANC8_decision": "Retained",
        "missing_handling": "Unknown retained; Not applicable retained structurally",
        "notes": "Not applicable corresponds to women not currently in union."
    },

    {
        "predictor_id": 6,
        "domain": "Maternal and sociodemographic",
        "construct": "Partner occupation",
        "source_variables": "v705",
        "final_variable": "partner_occupation_group",
        "final_type": "Nominal categorical",
        "final_coding": "Did not work; Professional/clerical; Sales/services; Agriculture; Manual/domestic; Unknown; Not applicable",
        "ANC8_decision": "Retained",
        "missing_handling": "Unknown retained; Not applicable retained structurally",
        "notes": "Not applicable corresponds to women not currently in union."
    },

    {
        "predictor_id": 7,
        "domain": "Maternal and sociodemographic",
        "construct": "Healthcare decision-making participation",
        "source_variables": "v743a",
        "final_variable": "healthcare_decision_participation",
        "final_type": "Categorical",
        "final_coding": "Participates; Does not participate; Not applicable",
        "ANC8_decision": "Retained",
        "missing_handling": "1 applicable unresolved response retained as true missing; structural Not applicable retained",
        "notes": "Participation refers to respondent involvement in decisions about her own healthcare."
    },

    {
        "predictor_id": 8,
        "domain": "Household and socioeconomic",
        "construct": "Household wealth quintile",
        "source_variables": "v190",
        "final_variable": "wealth_quintile",
        "final_type": "Ordinal categorical",
        "final_coding": "Poorest; Poorer; Middle; Richer; Richest",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": "National DHS wealth quintile."
    },

    {
        "predictor_id": 9,
        "domain": "Household and socioeconomic",
        "construct": "Place of residence",
        "source_variables": "v025",
        "final_variable": "residence",
        "final_type": "Binary categorical",
        "final_coding": "Urban; Rural",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": ""
    },

    {
        "predictor_id": 10,
        "domain": "Household and socioeconomic",
        "construct": "Region",
        "source_variables": "v024",
        "final_variable": "region",
        "final_type": "Nominal categorical",
        "final_coding": "14 EDHS regional categories",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": "All 14 regions retained."
    },

    {
        "predictor_id": 11,
        "domain": "Household and socioeconomic",
        "construct": "Household size",
        "source_variables": "v136",
        "final_variable": "household_size",
        "final_type": "Continuous/count",
        "final_coding": "Number of listed household members",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": "Observed range 1-39."
    },

    {
        "predictor_id": 12,
        "domain": "Household and socioeconomic",
        "construct": "Religion",
        "source_variables": "v130",
        "final_variable": "religion_group",
        "final_type": "Nominal categorical",
        "final_coding": "Orthodox; Protestant; Muslim; Other religion",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": "Catholic, Traditional and original Other combined as Other religion."
    },

    {
        "predictor_id": 13,
        "domain": "Household and socioeconomic",
        "construct": "Traditional media exposure",
        "source_variables": "v157; v158; v159",
        "final_variable": "traditional_media_weekly_count",
        "final_type": "Discrete count",
        "final_coding": "0-3 media types accessed at least weekly",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": "Newspaper, radio and television are source items only, not separate primary predictors."
    },

    {
        "predictor_id": 14,
        "domain": "Household and socioeconomic",
        "construct": "Internet exposure",
        "source_variables": "v171a",
        "final_variable": "internet_exposure",
        "final_type": "Nominal categorical",
        "final_coding": "Never used; Used in last 12 months; Used before last 12 months",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": ""
    },

    {
        "predictor_id": 15,
        "domain": "Household and socioeconomic",
        "construct": "Health insurance coverage",
        "source_variables": "v481",
        "final_variable": "health_insurance",
        "final_type": "Binary categorical",
        "final_coding": "No; Yes",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": ""
    },

    {
        "predictor_id": 16,
        "domain": "Pregnancy and obstetric",
        "construct": "Pregnancy order",
        "source_variables": "p32_01-p32_20",
        "final_variable": "pregnancy_order_group",
        "final_type": "Ordinal categorical",
        "final_coding": "First pregnancy; Pregnancy order 2-3; Pregnancy order 4-5; Pregnancy order 6+",
        "ANC8_decision": "ANC-specific amendment",
        "missing_handling": "No missing values",
        "notes": "Pregnancy-history count replaces live-birth order."
    },

    {
        "predictor_id": 17,
        "domain": "Pregnancy and obstetric",
        "construct": "Preceding pregnancy interval",
        "source_variables": "p11_01",
        "final_variable": "preceding_pregnancy_interval_group",
        "final_type": "Categorical",
        "final_coding": "First pregnancy; <24 months; 24-35 months; 36-59 months; 60+ months; Unknown/missing",
        "ANC8_decision": "ANC-specific amendment",
        "missing_handling": "First pregnancy retained structurally; 13 unresolved higher-order cases retained separately",
        "notes": "Pregnancy interval replaces preceding live-birth interval."
    },

    {
        "predictor_id": 18,
        "domain": "Pregnancy and obstetric",
        "construct": "Pregnancy intention",
        "source_variables": "m10_1",
        "final_variable": "pregnancy_intention",
        "final_type": "Ordinal categorical",
        "final_coding": "Wanted then; Wanted later; Wanted no more",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": ""
    },

    {
        "predictor_id": 19,
        "domain": "Pregnancy and obstetric",
        "construct": "Previous pregnancy outcome history",
        "source_variables": "p32_02-p32_20",
        "final_variable": "previous_pregnancy_history",
        "final_type": "Nominal categorical",
        "final_coding": "No previous pregnancy; Previous pregnancy, no adverse outcome; At least one previous adverse outcome",
        "ANC8_decision": "ANC-specific amendment",
        "missing_handling": "No unresolved records",
        "notes": "Index pregnancy position 01 excluded from previous-history construction."
    },

    {
        "predictor_id": 20,
        "domain": "Health service and access",
        "construct": "Travel time to nearest hospital or health center",
        "source_variables": "s1101",
        "final_variable": "travel_time_min",
        "final_type": "Continuous",
        "final_coding": "Minutes",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": "0 retained; 600 retained as top-coded 600+ minutes."
    },

    {
        "predictor_id": 21,
        "domain": "Health service and access",
        "construct": "Transport mode to nearest hospital or health center",
        "source_variables": "s1101a",
        "final_variable": "transport_mode_group",
        "final_type": "Nominal categorical",
        "final_coding": "Walking; Motorcycle/bajaj/taxi/minibus; Other motorized; Other non-motorized",
        "ANC8_decision": "Retained",
        "missing_handling": "No missing values",
        "notes": "Previously validated four-category grouping."
    }
])


# ---------------------------------------------------------------------
# Integrity checks against the ACTUAL fixed handoff dataset
# ---------------------------------------------------------------------

assert len(predictor_register) == 21
assert predictor_register["predictor_id"].is_unique
assert predictor_register["final_variable"].is_unique

assert set(predictor_register["final_variable"]) == set(anc8_primary_predictors), (
    "Predictor register does not exactly match the frozen 21-variable predictor list."
)

assert all(
    var in anc8_handoff.columns
    for var in predictor_register["final_variable"]
)


# ---------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------

predictor_register_path = (
    output_dir / "ANC8_predictor_register_final.csv"
)

predictor_register.to_csv(
    predictor_register_path,
    index=False
)


print("\n" + "=" * 90)
print("FINAL ANC8 PREDICTOR REGISTER")
print("=" * 90)

print("Rows:", len(predictor_register))
print("All variables found in frozen dataset: YES")
print("Exact match to 21-variable predictor list: YES")

print("\nSaved to:")
print(predictor_register_path)

print("\nPreview:")
print(
    predictor_register[
        [
            "predictor_id",
            "construct",
            "final_variable",
            "ANC8_decision"
        ]
    ].to_string(index=False)
)

In [ ]:
# =============================================================================
# EXPORT FINAL ANC8 OUTCOME REGISTER
#
# Purpose:
# Freeze the exact definitions and accounting for ANC4+ and ANC8+.
# =============================================================================

outcome_register = pd.DataFrame([
    {
        "outcome_id": 1,
        "outcome": "ANC4+ attainment",
        "source_variable": "m14_1",
        "final_variable": "anc4plus",
        "definition": "1 = four or more ANC contacts; 0 = fewer than four ANC contacts",
        "analytic_denominator": 6916,
        "resolved_n": 6893,
        "unresolved_n": 23,
        "unresolved_rule": "m14_1 = 98 (don't know) retained as missing",
        "zero_ANC_rule": "0 ANC contacts retained as outcome = 0",
        "unweighted_positive_n": 3234,
        "unweighted_prevalence_percent": 46.92,
        "survey_weighted_prevalence_percent": 52.43,
        "status": "Finalized"
    },

    {
        "outcome_id": 2,
        "outcome": "ANC8+ attainment",
        "source_variable": "m14_1",
        "final_variable": "anc8plus",
        "definition": "1 = eight or more ANC contacts; 0 = fewer than eight ANC contacts",
        "analytic_denominator": 6916,
        "resolved_n": 6893,
        "unresolved_n": 23,
        "unresolved_rule": "m14_1 = 98 (don't know) retained as missing",
        "zero_ANC_rule": "0 ANC contacts retained as outcome = 0",
        "unweighted_positive_n": 1173,
        "unweighted_prevalence_percent": 17.02,
        "survey_weighted_prevalence_percent": 19.16,
        "status": "Finalized"
    }
])

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(outcome_register) == 2

assert set(outcome_register["final_variable"]) == {
    "anc4plus", "anc8plus"
}

assert anc8_handoff["anc4plus"].isna().sum() == 23
assert anc8_handoff["anc8plus"].isna().sum() == 23

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

outcome_register_path = (
    output_dir / "ANC8_outcome_register_final.csv"
)

outcome_register.to_csv(
    outcome_register_path,
    index=False
)

print("\n" + "=" * 85)
print("FINAL ANC8 OUTCOME REGISTER")
print("=" * 85)

print(outcome_register.to_string(index=False))

print("\nSaved to:")
print(outcome_register_path)

In [ ]:
# =============================================================================
# TABLE 1 GENERATION - LOAD FROZEN ANC8 DATASET
#
# Purpose:
# Use only the fixed Notebook-1 analytic artifact as the source for Table 1.
#
# This ensures the Word table can be reproduced independently of the
# exploratory preprocessing objects still present in memory.
# =============================================================================

from pathlib import Path
import pandas as pd

output_dir = PROJECT / "outputs"

frozen_path = output_dir / "ANC8_analysis_frozen.pkl"

table1_df = pd.read_pickle(frozen_path)

print("Loaded:", frozen_path)
print("Shape:", table1_df.shape)

print("\nColumns:")
for i, col in enumerate(table1_df.columns, start=1):
    print(f"{i:>2}. {col}")

In [ ]:
# =============================================================================
# GENERATE PUBLICATION-READY ANC8 TABLE 1 AS A WORD DOCUMENT
#
# Source of truth:
#   ANC8_analysis_frozen.pkl
#
# Additional ANC-count descriptive artifacts:
#   ANC_contact_count_descriptive_summary.csv
#   ANC_contact_count_weighted_distribution.csv
#
# Output:
#   Table_1_ANC8_Characteristics.docx
#
# IMPORTANT:
# - No predictors are reconstructed.
# - No model preprocessing occurs here.
# - Categorical n values are unweighted sample counts.
# - Percentages and continuous summaries are survey weighted using v005.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.section import WD_SECTION
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


# =============================================================================
# 1. PATHS AND INPUTS
# =============================================================================

project_dir = PROJECT
output_dir = project_dir / "outputs"

frozen_path = output_dir / "ANC8_analysis_frozen.pkl"

anc_summary_path = (
    output_dir / "ANC_contact_count_descriptive_summary.csv"
)

anc_distribution_path = (
    output_dir / "ANC_contact_count_weighted_distribution.csv"
)

word_output_path = (
    output_dir / "Table_1_ANC8_Characteristics.docx"
)


# Load only the fixed analytic handoff dataset.
df = pd.read_pickle(frozen_path).copy()

assert df.shape == (6916, 27)
assert df["caseid"].is_unique
assert df["v005"].notna().all()


# DHS survey weight.
df["_weight"] = df["v005"] / 1_000_000


# =============================================================================
# 2. HELPER FUNCTIONS FOR SURVEY-WEIGHTED DESCRIPTIVES
# =============================================================================

def weighted_percent(data, variable, category, denominator_mask=None):
    """
    Survey-weighted percentage for one category.

    Parameters
    ----------
    data : DataFrame
    variable : str
        Categorical variable.
    category :
        Category whose percentage should be calculated.
    denominator_mask : boolean Series, optional
        Restricts the denominator when the construct is conditionally
        applicable, for example partner characteristics among currently
        partnered women.
    """

    if denominator_mask is None:
        denominator_mask = data[variable].notna()

    denominator_mask = denominator_mask & data["_weight"].notna()

    numerator_mask = (
        denominator_mask
        & data[variable].eq(category)
    )

    denominator_weight = data.loc[
        denominator_mask, "_weight"
    ].sum()

    numerator_weight = data.loc[
        numerator_mask, "_weight"
    ].sum()

    if denominator_weight == 0:
        return np.nan

    return numerator_weight / denominator_weight * 100


def weighted_mean(data, variable, mask=None):
    """Survey-weighted arithmetic mean."""

    if mask is None:
        mask = data[variable].notna()

    mask = (
        mask
        & data[variable].notna()
        & data["_weight"].notna()
    )

    x = data.loc[mask, variable].astype(float)
    w = data.loc[mask, "_weight"].astype(float)

    return np.average(x, weights=w)


def weighted_sd(data, variable, mask=None):
    """
    Descriptive survey-weighted standard deviation.

    This is a weighted descriptive SD rather than a design-based
    standard error.
    """

    if mask is None:
        mask = data[variable].notna()

    mask = (
        mask
        & data[variable].notna()
        & data["_weight"].notna()
    )

    x = data.loc[mask, variable].astype(float).to_numpy()
    w = data.loc[mask, "_weight"].astype(float).to_numpy()

    mean = np.average(x, weights=w)

    variance = np.average(
        (x - mean) ** 2,
        weights=w
    )

    return np.sqrt(variance)


def weighted_quantile(values, weights, quantiles):
    """
    Weighted empirical quantiles.

    Used for medians and interquartile ranges of skewed/count variables.
    """

    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)

    valid = (
        np.isfinite(values)
        & np.isfinite(weights)
        & (weights > 0)
    )

    values = values[valid]
    weights = weights[valid]

    order = np.argsort(values)

    values = values[order]
    weights = weights[order]

    cumulative_weight = np.cumsum(weights)
    cumulative_weight = cumulative_weight / cumulative_weight[-1]

    return np.interp(
        quantiles,
        cumulative_weight,
        values
    )


def weighted_median_iqr(data, variable, mask=None):
    """Return weighted median, Q1 and Q3."""

    if mask is None:
        mask = data[variable].notna()

    mask = (
        mask
        & data[variable].notna()
        & data["_weight"].notna()
    )

    q1, median, q3 = weighted_quantile(
        data.loc[mask, variable].astype(float),
        data.loc[mask, "_weight"].astype(float),
        [0.25, 0.50, 0.75]
    )

    return median, q1, q3


# =============================================================================
# 3. TABLE-BUILDING HELPERS
# =============================================================================

table_rows = []


def add_section(title):
    """Add a domain heading row."""

    table_rows.append({
        "type": "section",
        "characteristic": title,
        "category": "",
        "n": "",
        "estimate": ""
    })


def add_categorical(
    characteristic,
    variable,
    categories,
    denominator_mask=None,
    display_labels=None
):
    """
    Add one categorical characteristic.

    categories controls the display order and prevents automatic
    alphabetical ordering.
    """

    if display_labels is None:
        display_labels = {
            category: str(category)
            for category in categories
        }

    for i, category in enumerate(categories):

        if denominator_mask is None:
            mask = df[variable].notna()
        else:
            mask = denominator_mask & df[variable].notna()

        n = int(
            (
                mask
                & df[variable].eq(category)
            ).sum()
        )

        pct = weighted_percent(
            df,
            variable,
            category,
            denominator_mask=mask
        )

        table_rows.append({
            "type": "data",
            "characteristic":
                characteristic if i == 0 else "",
            "category":
                display_labels.get(category, str(category)),
            "n": f"{n}",
            "estimate": f"{pct:.1f}%"
        })


def add_continuous_mean_sd(
    characteristic,
    variable,
    decimals=1,
    mask=None
):
    """Add weighted mean +/- descriptive weighted SD."""

    if mask is None:
        mask = df[variable].notna()

    n = int(
        (
            mask
            & df[variable].notna()
        ).sum()
    )

    mean = weighted_mean(
        df,
        variable,
        mask=mask
    )

    sd = weighted_sd(
        df,
        variable,
        mask=mask
    )

    table_rows.append({
        "type": "data",
        "characteristic": characteristic,
        "category": "",
        "n": f"{n}",
        "estimate":
            f"{mean:.{decimals}f} ± {sd:.{decimals}f}"
    })


def add_continuous_median_iqr(
    characteristic,
    variable,
    decimals=0,
    mask=None
):
    """Add survey-weighted median (IQR)."""

    if mask is None:
        mask = df[variable].notna()

    n = int(
        (
            mask
            & df[variable].notna()
        ).sum()
    )

    median, q1, q3 = weighted_median_iqr(
        df,
        variable,
        mask=mask
    )

    table_rows.append({
        "type": "data",
        "characteristic": characteristic,
        "category": "",
        "n": f"{n}",
        "estimate":
            f"{median:.{decimals}f} "
            f"({q1:.{decimals}f}–{q3:.{decimals}f})"
    })


# =============================================================================
# 4. DEFINE SPECIAL DENOMINATORS
# =============================================================================

currently_in_union = (
    df["maternal_current_union_status"]
    .eq("Currently in union")
)

known_decision = (
    currently_in_union
    & df["healthcare_decision_participation"].notna()
    & ~df["healthcare_decision_participation"]
        .eq("Not applicable")
)

known_preceding_interval = (
    ~df["preceding_pregnancy_interval_group"]
        .isin(["First pregnancy", "Unknown / missing"])
)

known_anc4 = df["anc4plus"].notna()
known_anc8 = df["anc8plus"].notna()


# =============================================================================
# 5. BUILD TABLE CONTENT
# =============================================================================

# ---------------------------------------------------------------------
# Maternal and sociodemographic
# ---------------------------------------------------------------------

add_section(
    "Maternal and sociodemographic characteristics"
)

add_continuous_mean_sd(
    "Maternal age at interview, years",
    "v012",
    decimals=1
)

add_categorical(
    "Maternal education",
    "maternal_education",
    [
        "No education",
        "Primary",
        "Secondary",
        "Higher"
    ]
)

add_categorical(
    "Current union status",
    "maternal_current_union_status",
    [
        "Currently in union",
        "Not currently in union"
    ]
)

add_categorical(
    "Maternal occupation",
    "maternal_occupation_group",
    [
        "Not working",
        "Professional/clerical",
        "Sales/services",
        "Agriculture",
        "Manual/domestic"
    ]
)

add_categorical(
    "Husband/partner education",
    "partner_education",
    [
        "No education",
        "Primary",
        "Secondary",
        "Higher",
        "Unknown"
    ],
    denominator_mask=currently_in_union
)

add_categorical(
    "Husband/partner occupation",
    "partner_occupation_group",
    [
        "Did not work",
        "Professional/clerical",
        "Sales/services",
        "Agriculture",
        "Manual/domestic",
        "Unknown"
    ],
    denominator_mask=currently_in_union
)

add_categorical(
    "Participation in healthcare decision-making",
    "healthcare_decision_participation",
    [
        "Participates",
        "Does not participate"
    ],
    denominator_mask=known_decision
)


# ---------------------------------------------------------------------
# Household and socioeconomic
# ---------------------------------------------------------------------

add_section(
    "Household and socioeconomic characteristics"
)

add_categorical(
    "Household wealth quintile",
    "wealth_quintile",
    [
        "Poorest",
        "Poorer",
        "Middle",
        "Richer",
        "Richest"
    ]
)

add_categorical(
    "Place of residence",
    "residence",
    [
        "Urban",
        "Rural"
    ]
)

add_categorical(
    "Region",
    "region",
    [
        "Tigray",
        "Afar",
        "Amhara",
        "Oromia",
        "Somali",
        "Benishangul-Gumuz",
        "Central Ethiopia",
        "Sidama",
        "South West Ethiopia",
        "South Ethiopia",
        "Gambella",
        "Harari",
        "Addis Ababa",
        "Dire Dawa"
    ]
)

add_continuous_median_iqr(
    "Household size, members",
    "household_size",
    decimals=0
)

add_categorical(
    "Religion",
    "religion_group",
    [
        "Orthodox",
        "Protestant",
        "Muslim",
        "Other religion"
    ]
)

add_categorical(
    "Weekly traditional media exposure",
    "traditional_media_weekly_count",
    [0, 1, 2, 3],
    display_labels={
        0: "No media accessed at least weekly",
        1: "One medium accessed at least weekly",
        2: "Two media accessed at least weekly",
        3: "All three media accessed at least weekly"
    }
)

add_categorical(
    "Internet exposure",
    "internet_exposure",
    [
        "Never used",
        "Used before last 12 months",
        "Used in last 12 months"
    ]
)

add_categorical(
    "Health insurance coverage",
    "health_insurance",
    [
        "No",
        "Yes"
    ],
    display_labels={
        "No": "Not insured",
        "Yes": "Insured"
    }
)


# ---------------------------------------------------------------------
# Pregnancy and obstetric
# ---------------------------------------------------------------------

add_section(
    "Pregnancy and obstetric characteristics"
)

add_categorical(
    "Pregnancy order",
    "pregnancy_order_group",
    [
        "First pregnancy",
        "Pregnancy order 2-3",
        "Pregnancy order 4-5",
        "Pregnancy order 6+"
    ],
    display_labels={
        "First pregnancy": "First pregnancy",
        "Pregnancy order 2-3": "Second/third pregnancy",
        "Pregnancy order 4-5": "Fourth/fifth pregnancy",
        "Pregnancy order 6+": "Sixth or higher pregnancy"
    }
)

add_categorical(
    "Preceding pregnancy interval",
    "preceding_pregnancy_interval_group",
    [
        "<24 months",
        "24-35 months",
        "36-59 months",
        "60+ months"
    ],
    denominator_mask=known_preceding_interval
)

add_categorical(
    "Pregnancy intention",
    "pregnancy_intention",
    [
        "Wanted then",
        "Wanted later",
        "Wanted no more"
    ]
)

add_categorical(
    "Previous pregnancy outcome history",
    "previous_pregnancy_history",
    [
        "No previous pregnancy",
        "Previous pregnancy, no adverse outcome",
        "At least one previous adverse outcome"
    ]
)


# ---------------------------------------------------------------------
# Health service access
# ---------------------------------------------------------------------

add_section(
    "Health service and access characteristics"
)

add_continuous_median_iqr(
    "Travel time to nearest hospital or health center, minutes",
    "travel_time_min",
    decimals=0
)

add_categorical(
    "Transport mode to nearest hospital or health center",
    "transport_mode_group",
    [
        "Walking",
        "Motorcycle/bajaj/taxi/minibus",
        "Other motorized",
        "Other non-motorized"
    ]
)


# ---------------------------------------------------------------------
# ANC attainment
# ---------------------------------------------------------------------

add_section(
    "Antenatal care attainment"
)

# ANC contact-count summary from the already-exported Notebook-1 artifact.
anc_summary = pd.read_csv(anc_summary_path).iloc[0]

# Calculate weighted median/IQR from the exported weighted distribution.
anc_dist = pd.read_csv(anc_distribution_path).copy()

anc_values = anc_dist["anc_contacts"].to_numpy(dtype=float)
anc_weights = anc_dist["weighted_sum"].to_numpy(dtype=float)

anc_q1, anc_median, anc_q3 = weighted_quantile(
    anc_values,
    anc_weights,
    [0.25, 0.50, 0.75]
)

table_rows.append({
    "type": "data",
    "characteristic": "Number of ANC contacts",
    "category": "",
    "n": f"{int(anc_summary['resolved_n'])}",
    "estimate":
        f"{anc_median:.0f} "
        f"({anc_q1:.0f}–{anc_q3:.0f})"
})


add_categorical(
    "ANC4+ attainment",
    "anc4plus",
    [0.0, 1.0],
    denominator_mask=known_anc4,
    display_labels={
        0.0: "Fewer than four ANC contacts",
        1.0: "Four or more ANC contacts"
    }
)

add_categorical(
    "ANC8+ attainment",
    "anc8plus",
    [0.0, 1.0],
    denominator_mask=known_anc8,
    display_labels={
        0.0: "Fewer than eight ANC contacts",
        1.0: "Eight or more ANC contacts"
    }
)


# Convert to dataframe for optional inspection/export.
table1_content = pd.DataFrame(table_rows)


# =============================================================================
# 6. WORD FORMATTING HELPERS
# =============================================================================

def set_cell_shading(cell, fill):
    """Apply background shading to a Word table cell."""

    tc_pr = cell._tc.get_or_add_tcPr()

    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)

    tc_pr.append(shd)


def set_cell_margins(
    cell,
    top=70,
    start=90,
    bottom=70,
    end=90
):
    """Set internal cell margins in twentieths of a point."""

    tc = cell._tc
    tc_pr = tc.get_or_add_tcPr()

    tc_mar = tc_pr.first_child_found_in(
        "w:tcMar"
    )

    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar")
        tc_pr.append(tc_mar)

    for margin_name, margin_value in {
        "top": top,
        "start": start,
        "bottom": bottom,
        "end": end
    }.items():

        node = tc_mar.find(
            qn(f"w:{margin_name}")
        )

        if node is None:
            node = OxmlElement(
                f"w:{margin_name}"
            )
            tc_mar.append(node)

        node.set(
            qn("w:w"),
            str(margin_value)
        )

        node.set(
            qn("w:type"),
            "dxa"
        )


def set_repeat_table_header(row):
    """Repeat the table header row on subsequent pages."""

    tr_pr = row._tr.get_or_add_trPr()

    tbl_header = OxmlElement("w:tblHeader")
    tbl_header.set(
        qn("w:val"),
        "true"
    )

    tr_pr.append(tbl_header)


def remove_table_borders(table):
    """
    Remove full grid borders so the table resembles a publication table
    rather than a spreadsheet.
    """

    tbl = table._tbl
    tbl_pr = tbl.tblPr

    borders = tbl_pr.first_child_found_in(
        "w:tblBorders"
    )

    if borders is None:
        borders = OxmlElement("w:tblBorders")
        tbl_pr.append(borders)

    for edge in [
        "top",
        "left",
        "bottom",
        "right",
        "insideH",
        "insideV"
    ]:

        tag = "start" if edge == "left" else (
            "end" if edge == "right" else edge
        )

        element = OxmlElement(
            f"w:{tag}"
        )

        element.set(
            qn("w:val"),
            "nil"
        )

        borders.append(element)


def set_bottom_border(cell, size=8):
    """Add a bottom border to a cell."""

    tc_pr = cell._tc.get_or_add_tcPr()

    borders = tc_pr.first_child_found_in(
        "w:tcBorders"
    )

    if borders is None:
        borders = OxmlElement("w:tcBorders")
        tc_pr.append(borders)

    bottom = OxmlElement("w:bottom")

    bottom.set(qn("w:val"), "single")
    bottom.set(qn("w:sz"), str(size))
    bottom.set(qn("w:space"), "0")
    bottom.set(qn("w:color"), "000000")

    borders.append(bottom)


# =============================================================================
# 7. CREATE WORD DOCUMENT
# =============================================================================

doc = Document()


# ---------------------------------------------------------------------
# Page setup
# ---------------------------------------------------------------------

section = doc.sections[0]

section.top_margin = Inches(0.55)
section.bottom_margin = Inches(0.55)
section.left_margin = Inches(0.55)
section.right_margin = Inches(0.55)


# ---------------------------------------------------------------------
# Default font
# ---------------------------------------------------------------------

styles = doc.styles

styles["Normal"].font.name = "Arial"
styles["Normal"].font.size = Pt(9)


# ---------------------------------------------------------------------
# Title
# ---------------------------------------------------------------------

title = doc.add_paragraph()

title.alignment = WD_ALIGN_PARAGRAPH.CENTER

run = title.add_run(
    "Table 1. Characteristics of the ANC8 analytic sample"
)

run.bold = True
run.font.name = "Arial"
run.font.size = Pt(11)


# ---------------------------------------------------------------------
# Main table
# ---------------------------------------------------------------------

table = doc.add_table(
    rows=1,
    cols=4
)

table.alignment = WD_TABLE_ALIGNMENT.CENTER
table.autofit = False

remove_table_borders(table)


# Column widths
widths = [
    Inches(2.65),
    Inches(2.75),
    Inches(0.90),
    Inches(1.55)
]

header_cells = table.rows[0].cells

headers = [
    "Characteristic",
    "Category",
    "Unweighted n",
    "Survey-weighted estimate"
]

for i, (cell, text) in enumerate(
    zip(header_cells, headers)
):

    cell.width = widths[i]
    cell.vertical_alignment = (
        WD_CELL_VERTICAL_ALIGNMENT.CENTER
    )

    set_cell_margins(cell)

    paragraph = cell.paragraphs[0]
    paragraph.alignment = (
        WD_ALIGN_PARAGRAPH.LEFT
        if i < 2
        else WD_ALIGN_PARAGRAPH.CENTER
    )

    run = paragraph.add_run(text)

    run.bold = True
    run.font.name = "Arial"
    run.font.size = Pt(9)

    set_bottom_border(cell, size=10)


set_repeat_table_header(
    table.rows[0]
)


# ---------------------------------------------------------------------
# Populate table
# ---------------------------------------------------------------------

for _, row in table1_content.iterrows():

    cells = table.add_row().cells

    for i, cell in enumerate(cells):
        cell.width = widths[i]
        cell.vertical_alignment = (
            WD_CELL_VERTICAL_ALIGNMENT.CENTER
        )

        set_cell_margins(cell)

    if row["type"] == "section":

        # Merge the four cells for a domain heading.
        merged = cells[0]

        for cell in cells[1:]:
            merged = merged.merge(cell)

        set_cell_shading(
            merged,
            "E7E6E6"
        )

        paragraph = merged.paragraphs[0]

        paragraph.paragraph_format.space_before = Pt(2)
        paragraph.paragraph_format.space_after = Pt(2)

        run = paragraph.add_run(
            str(row["characteristic"])
        )

        run.bold = True
        run.font.name = "Arial"
        run.font.size = Pt(9)

        continue


    values = [
        row["characteristic"],
        row["category"],
        row["n"],
        row["estimate"]
    ]

    for i, value in enumerate(values):

        paragraph = cells[i].paragraphs[0]

        paragraph.paragraph_format.space_before = Pt(0)
        paragraph.paragraph_format.space_after = Pt(0)

        paragraph.alignment = (
            WD_ALIGN_PARAGRAPH.LEFT
            if i < 2
            else WD_ALIGN_PARAGRAPH.CENTER
        )

        run = paragraph.add_run(
            str(value)
        )

        run.font.name = "Arial"
        run.font.size = Pt(8.5)


# =============================================================================
# 8. TABLE NOTES
# =============================================================================

doc.add_paragraph()

note = doc.add_paragraph()

note.paragraph_format.space_before = Pt(2)
note.paragraph_format.space_after = Pt(0)

run = note.add_run("Note. ")
run.bold = True
run.font.name = "Arial"
run.font.size = Pt(8)

run = note.add_run(
    "Values for categorical characteristics are unweighted sample counts "
    "with survey-weighted percentages. "
    "Continuous characteristics are reported as survey-weighted mean ± "
    "descriptive weighted standard deviation or survey-weighted median "
    "(interquartile range), as specified."
)

run.font.name = "Arial"
run.font.size = Pt(8)


# Partner denominator note
p = doc.add_paragraph()

p.paragraph_format.space_before = Pt(0)
p.paragraph_format.space_after = Pt(0)

r = p.add_run(
    "Husband/partner education and occupation are reported among women "
    f"currently in union (N={int(currently_in_union.sum()):,}). "
    "Healthcare decision-making participation is reported among currently "
    f"partnered women with known decision-making status "
    f"(N={int(known_decision.sum()):,})."
)

r.font.name = "Arial"
r.font.size = Pt(8)


# Pregnancy interval denominator
p = doc.add_paragraph()

p.paragraph_format.space_before = Pt(0)
p.paragraph_format.space_after = Pt(0)

r = p.add_run(
    "Preceding pregnancy interval is reported among women with a known "
    f"applicable interval (N={int(known_preceding_interval.sum()):,}); "
    "first pregnancies and 13 unresolved higher-order pregnancy intervals "
    "are excluded from that descriptive denominator."
)

r.font.name = "Arial"
r.font.size = Pt(8)


# Outcome denominator and count summary
p = doc.add_paragraph()

p.paragraph_format.space_before = Pt(0)
p.paragraph_format.space_after = Pt(0)

r = p.add_run(
    "ANC contact count, ANC4+, and ANC8+ are reported among women with a "
    f"resolved number of ANC contacts (N={int(known_anc8.sum()):,}); "
    "23 responses coded 98 (don't know) are unresolved. "
    f"The survey-weighted mean number of ANC contacts was "
    f"{float(anc_summary['survey_weighted_mean']):.2f}."
)

r.font.name = "Arial"
r.font.size = Pt(8)


# Final rounding note
p = doc.add_paragraph()

p.paragraph_format.space_before = Pt(0)
p.paragraph_format.space_after = Pt(0)

r = p.add_run(
    "Percentages may not total exactly 100.0 because of rounding."
)

r.font.name = "Arial"
r.font.size = Pt(8)


# =============================================================================
# 9. SAVE WORD DOCUMENT
# =============================================================================

doc.save(word_output_path)


print("\n" + "=" * 90)
print("ANC8 TABLE 1 WORD EXPORT")
print("=" * 90)

print("Source frozen dataset:")
print(frozen_path)

print("\nRows in source dataset:")
print(len(df))

print("\nWord document saved to:")
print(word_output_path)

print("\nTable rows generated:")
print(len(table1_content))

print("\nSpecial denominators:")
print(
    "Currently in union:",
    int(currently_in_union.sum())
)
print(
    "Known healthcare decision participation:",
    int(known_decision.sum())
)
print(
    "Known applicable preceding pregnancy interval:",
    int(known_preceding_interval.sum())
)
print(
    "Resolved ANC outcomes:",
    int(known_anc8.sum())
)

print("\nDone.")

In [ ]:
# =============================================================================
# FINAL REFRESH OF ANC8 NOTEBOOK-1 ARTIFACT MANIFEST
#
# Purpose:
# Create the definitive inventory of all Notebook-1 outputs after
# predictor, outcome, descriptive, visual, and Word-table exports.
# =============================================================================

from pathlib import Path
import hashlib
import pandas as pd

output_dir = PROJECT / "outputs"


def sha256_file(path):
    """Return SHA-256 hash for one file."""
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)

    return h.hexdigest()


final_artifacts = [
    {
        "artifact": "Frozen analytic dataset",
        "filename": "ANC8_analysis_frozen.pkl",
        "purpose": "Authoritative Notebook-2 modelling input",
        "rows": 6916,
        "columns": 27
    },
    {
        "artifact": "Final predictor register",
        "filename": "ANC8_predictor_register_final.csv",
        "purpose": "Documents the exact 21 frozen predictors",
        "rows": 21,
        "columns": None
    },
    {
        "artifact": "Final outcome register",
        "filename": "ANC8_outcome_register_final.csv",
        "purpose": "Documents ANC4+ and ANC8+ outcome definitions and accounting",
        "rows": 2,
        "columns": None
    },
    {
        "artifact": "ANC contact descriptive summary",
        "filename": "ANC_contact_count_descriptive_summary.csv",
        "purpose": "Summary statistics for number of ANC contacts",
        "rows": 1,
        "columns": None
    },
    {
        "artifact": "ANC contact weighted distribution",
        "filename": "ANC_contact_count_weighted_distribution.csv",
        "purpose": "Survey-weighted exact ANC contact-count distribution",
        "rows": None,
        "columns": None
    },
    {
        "artifact": "ANC contact distribution figure",
        "filename": "Figure_ANC_contact_count_weighted_distribution.png",
        "purpose": "High-level survey-weighted ANC contact-count visual",
        "rows": None,
        "columns": None
    },
    {
        "artifact": "ANC8 Table 1",
        "filename": "Table_1_ANC8_Characteristics.docx",
        "purpose": "Publication-ready descriptive characteristics table",
        "rows": None,
        "columns": None
    }
]


manifest_rows = []

for item in final_artifacts:

    path = output_dir / item["filename"]

    manifest_rows.append({
        **item,
        "exists": path.exists(),
        "sha256": sha256_file(path) if path.exists() else None
    })


final_manifest = pd.DataFrame(manifest_rows)

final_manifest_path = (
    output_dir / "ANC8_artifact_manifest_final.csv"
)

final_manifest.to_csv(
    final_manifest_path,
    index=False
)


print("\n" + "=" * 100)
print("FINAL ANC8 NOTEBOOK-1 ARTIFACT MANIFEST")
print("=" * 100)

print(
    final_manifest.to_string(index=False)
)

print("\nAll required artifacts present:",
      final_manifest["exists"].all())

print("\nFinal manifest saved to:")
print(final_manifest_path)

## Handing on to the next notebooks

The 21 predictors and how each is typed, in the form the later notebooks read. The
typing (four numbers, three yes/no, fourteen categories) is the study's; the list
must be the one this notebook just built.

In [ ]:
arch = repro.reference("predictor_architecture.json")
repro.check("the 21 predictors are the study's, in the study's order",
            list(anc8_primary_predictors) == arch["full_cohort_predictors"])
handoff = PROJECT / "outputs" / "notebook3_explainability_handoff"
handoff.mkdir(parents=True, exist_ok=True)
(handoff / "predictor_architecture.json").write_text(json.dumps(arch, indent=2))
print(len(arch["full_cohort_predictors"]), "predictors written to", handoff / "predictor_architecture.json")

## Checks against the study

In [ ]:
ref = repro.reference("fingerprints.json")
frame = pd.read_pickle(PROJECT / "outputs" / "ANC8_analysis_frozen.pkl")
repro.check("6,916 maternal-health records, 23 of them without a resolved count", len(frame) == 6916)
repro.check("the analysis frame is identical to the study's (content fingerprint)",
            repro.fingerprint(frame) == ref["analysis_frame_21"], repro.fingerprint(frame))
repro.report()